# OneDrive Duplicate Cleaner

This notebook detects duplicate files using **SHA-256 hashing**, can delete duplicates, and rename cryptic image filenames based on metadata (EXIF).

Important safety notes:
- Always test first with `DRY_RUN = True`.
- Only afterwards set `DELETE_DUPLICATES = True`.
- Create a backup before the first real run.

In [1]:
from __future__ import annotations

# Standard libraries
import csv
import hashlib
import os
import re
import shutil
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional

# Pillow is used to read image metadata (EXIF)
# If it is not installed, the notebook still runs,
# but EXIF date values cannot be read.
try:
    from PIL import Image
except ImportError:
    Image = None
    print("Note: Pillow is not installed. EXIF reading is disabled. Install with: pip install pillow")


In [2]:
# === Configuration ===
# This notebook is preview-first and safety-oriented.
# You can always inspect plans/logs before allowing live file changes.

# ROOT_DIR
# What: Root folder scanned recursively by all phases.
# Example: Path(r"X:/Privat/Dublicatefinder (Onedrive)")
ROOT_DIR = Path.home() / "OneDrive"

# DRY_RUN
# True : Preview only. No destructive file changes are executed.
# False: Live execution can run, but only if confirmation gates are valid.
# Example: DRY_RUN = True
DRY_RUN = True

# DELETE_DUPLICATES
# True : Phase 2 can delete exact duplicate files (after safety gate check).
# False: No duplicate deletion execution.
# Example: DELETE_DUPLICATES = False
DELETE_DUPLICATES = False

# RENAME_IMAGES
# True : Phase 3 can execute renames after preview + safety gate.
# False: No live renames; preview can still run if preview override is enabled.
# Example: RENAME_IMAGES = True
RENAME_IMAGES = True

# FIND_SIMILAR_IMAGES
# True : Enable near-duplicate image detection (dHash).
# False: Skip near-duplicate detection phase.
# Example: FIND_SIMILAR_IMAGES = True
FIND_SIMILAR_IMAGES = True

# FIND_AI_VISUAL_GROUPS
# True : Enable CLIP-based visual similarity grouping.
# False: Skip AI visual grouping phase.
# Example: FIND_AI_VISUAL_GROUPS = True
FIND_AI_VISUAL_GROUPS = True

# MOVE_AI_GROUPS_TO_QUARANTINE
# True : Phase 1D can move AI-similar files to quarantine (keep one file per group).
# False: Only quarantine plan/preview, no move execution.
# Example: MOVE_AI_GROUPS_TO_QUARANTINE = False
MOVE_AI_GROUPS_TO_QUARANTINE = False

# AI_QUARANTINE_DIR
# What: Target folder for quarantine moves.
# Example: ROOT_DIR / "_duplicate_cleaner_quarantine" / "ai_visual"
AI_QUARANTINE_DIR = ROOT_DIR / "_duplicate_cleaner_quarantine" / "ai_visual"

# VERBOSE
# True : Print timestamped logs.
# False: Suppress most logs.
# Example: VERBOSE = True
VERBOSE = True

# LOG_EVERY_FILES
# What: Print progress log every N files in long loops.
# Example: LOG_EVERY_FILES = 500
LOG_EVERY_FILES = 500

# LOG_EVERY_BUCKETS
# What: Print progress every N buckets in grouped operations.
# Example: LOG_EVERY_BUCKETS = 50
LOG_EVERY_BUCKETS = 50

# SHOW_PREVIEW_DETAILS
# True : Print detailed preview entries (group contents, rename lines, etc.).
# False: Print only compact summary lines.
# Example: SHOW_PREVIEW_DETAILS = True
SHOW_PREVIEW_DETAILS = True

# PREVIEW_MAX_DUPLICATE_GROUPS
# What: Max printed duplicate groups in preview.
# 0 or negative => print all groups.
# Example: PREVIEW_MAX_DUPLICATE_GROUPS = 25
PREVIEW_MAX_DUPLICATE_GROUPS = 0

# PREVIEW_MAX_RENAMES
# What: Max printed rename actions in preview.
# 0 or negative => print all actions.
# Example: PREVIEW_MAX_RENAMES = 100
PREVIEW_MAX_RENAMES = 0

# PREVIEW_MAX_SIMILAR_GROUPS
# What: Max printed near-duplicate groups in preview.
# 0 or negative => print all groups.
# Example: PREVIEW_MAX_SIMILAR_GROUPS = 50
PREVIEW_MAX_SIMILAR_GROUPS = 0

# PREVIEW_MAX_AI_GROUPS
# What: Max printed AI visual groups in preview.
# 0 or negative => print all groups.
# Example: PREVIEW_MAX_AI_GROUPS = 20
PREVIEW_MAX_AI_GROUPS = 0

# PREVIEW_MAX_AI_QUARANTINE
# What: Max printed quarantine move actions in preview.
# 0 or negative => print all actions.
# Example: PREVIEW_MAX_AI_QUARANTINE = 200
PREVIEW_MAX_AI_QUARANTINE = 0

# SIMILAR_HASH_THRESHOLD
# What: dHash Hamming distance threshold (0..64) for near-duplicates.
# Lower => stricter, fewer matches.
# Example: SIMILAR_HASH_THRESHOLD = 6
SIMILAR_HASH_THRESHOLD = 6

# SIMILAR_NEIGHBOR_WINDOW
# What: Compare each image with next N time-neighbors in same resolution bucket.
# Higher => more comparisons, higher recall, slower runtime.
# Example: SIMILAR_NEIGHBOR_WINDOW = 20
SIMILAR_NEIGHBOR_WINDOW = 20

# SIMILAR_MIN_FILE_SIZE_BYTES
# What: Skip very small images in near-duplicate detection.
# Example: SIMILAR_MIN_FILE_SIZE_BYTES = 20000
SIMILAR_MIN_FILE_SIZE_BYTES = 20000

# AI_MODEL_NAME
# What: CLIP model for AI visual similarity.
# Example: AI_MODEL_NAME = "openai/clip-vit-base-patch32"
AI_MODEL_NAME = "openai/clip-vit-base-patch32"

# AI_SIMILARITY_THRESHOLD
# What: Fallback similarity threshold in [0..1], used when AI_SIMILARITY_PERCENT is None.
# Higher => stricter matching.
# Example: AI_SIMILARITY_THRESHOLD = 0.93
AI_SIMILARITY_THRESHOLD = 0.93

# AI_SIMILARITY_PERCENT
# What: Preferred threshold input in percent [0..100].
# If set, this overrides AI_SIMILARITY_THRESHOLD.
# Example: AI_SIMILARITY_PERCENT = 92.0
AI_SIMILARITY_PERCENT = 92.0

# AI_NEIGHBOR_WINDOW
# What: Compare each embedding with next N neighbors when not exhaustive.
# Ignored when AI_COMPARE_ALL_PAIRS=True.
# Example: AI_NEIGHBOR_WINDOW = 60
AI_NEIGHBOR_WINDOW = 60

# AI_MIN_FILE_SIZE_BYTES
# What: Skip very small images in AI visual grouping.
# Example: AI_MIN_FILE_SIZE_BYTES = 20000
AI_MIN_FILE_SIZE_BYTES = 20000

# AI_BATCH_SIZE
# What: Embedding batch size for CLIP inference.
# Higher => faster on GPU, higher memory usage.
# Example: AI_BATCH_SIZE = 16
AI_BATCH_SIZE = 16

# AI_DEVICE
# What: Preferred device for AI workloads.
# "auto" => use CUDA when available, otherwise CPU
# "cuda" => force CUDA (falls back to CPU with warning if unavailable)
# "cpu"  => force CPU
# Example: AI_DEVICE = "auto"
AI_DEVICE = "auto"

# AI_USE_FP16_ON_CUDA
# True : Use float16 on CUDA for faster inference and lower VRAM.
# False: Use float32 on CUDA.
# Example: AI_USE_FP16_ON_CUDA = True
AI_USE_FP16_ON_CUDA = True

# AI_ENABLE_TF32
# True : Enable TF32 on supported NVIDIA GPUs (faster matmul/convolution).
# False: Keep TF32 disabled.
# Example: AI_ENABLE_TF32 = True
AI_ENABLE_TF32 = True

# AI_COMPARE_ALL_PAIRS
# True : Exhaustive pair comparison (O(n^2)); maximum recall, potentially very slow.
# False: Windowed comparison using AI_NEIGHBOR_WINDOW; faster, may miss far-apart matches.
# Example: AI_COMPARE_ALL_PAIRS = True
AI_COMPARE_ALL_PAIRS = True

# AI_VERBOSE
# True : Detailed AI phase logs (scan/batch/pair progress).
# False: Compact AI logs.
# Example: AI_VERBOSE = True
AI_VERBOSE = True

# AI_VERBOSE_PAIR_LOG_LIMIT
# What: Log first N matching AI pairs in detail.
# Example: AI_VERBOSE_PAIR_LOG_LIMIT = 25
AI_VERBOSE_PAIR_LOG_LIMIT = 25

# RENAME_WITH_AI_LABEL
# True : Add AI-generated semantic label to timestamp-based rename target.
# False: Timestamp-only rename targets.
# Example: RENAME_WITH_AI_LABEL = True
RENAME_WITH_AI_LABEL = True

# AI_RENAME_MODEL_NAME
# What: Primary image caption model for semantic rename labels.
# Example: AI_RENAME_MODEL_NAME = "Salesforce/blip-image-captioning-large"
AI_RENAME_MODEL_NAME = "Salesforce/blip-image-captioning-large"

# AI_RENAME_MODEL_FALLBACK
# What: Fallback model if primary caption model cannot load.
# Example: AI_RENAME_MODEL_FALLBACK = "Salesforce/blip-image-captioning-base"
AI_RENAME_MODEL_FALLBACK = "Salesforce/blip-image-captioning-base"

# AI_RENAME_MAX_WORDS
# What: Word budget for AI label.
# >0 => CamelCase label with max words.
#  0 => full mini-sentence label (underscore style).
# Example: AI_RENAME_MAX_WORDS = 8
AI_RENAME_MAX_WORDS = 0

# AI_RENAME_MAX_CHARS
# What: Maximum label length to keep filenames manageable.
# Example: AI_RENAME_MAX_CHARS = 120
AI_RENAME_MAX_CHARS = 250

# AI_RENAME_DATE_STYLE
# What: Date format used at the start of renamed image filenames.
# "english_text" => "13. Feb.2026"
# "iso"          => "2026-02-13"
# Example: AI_RENAME_DATE_STYLE = "english_text"
AI_RENAME_DATE_STYLE = "english_text"

# AI_RENAME_INCLUDE_TIME
# True : Append "_HH-MM-SS" after the date.
# False: Use date-only prefix.
# Example: AI_RENAME_INCLUDE_TIME = False
AI_RENAME_INCLUDE_TIME = False

# AI_RENAME_USE_CAPTION_SENTENCE
# True : Use the AI caption sentence in the filename (human-readable text).
# False: Use compact AI label token style.
# Example: AI_RENAME_USE_CAPTION_SENTENCE = True
AI_RENAME_USE_CAPTION_SENTENCE = True

# AI_SUPPRESS_MODEL_LOAD_WARNINGS
# True : Suppress noisy model load reports/warnings from Hugging Face/Transformers.
# False: Show full model load diagnostics.
# Example: AI_SUPPRESS_MODEL_LOAD_WARNINGS = True
AI_SUPPRESS_MODEL_LOAD_WARNINGS = True

# AI_ENRICH_DATE_ONLY_NAMES
# True : Also enrich already date-only names with AI labels.
# False: Keep date-only names untouched unless other rename rules apply.
# Example: AI_ENRICH_DATE_ONLY_NAMES = True
AI_ENRICH_DATE_ONLY_NAMES = True

# RENAME_ALL_IMAGES_WITH_AI_LABEL
# True : Rename all image files (not only cryptic/date-like names).
# False: Rename only cryptic names (plus optional date-only enrichment rule).
# Example: RENAME_ALL_IMAGES_WITH_AI_LABEL = False
RENAME_ALL_IMAGES_WITH_AI_LABEL = False

# RENAME_VERBOSE_SUGGESTIONS
# True : Log full-path rename suggestions with current/proposed names and AI caption.
# False: Use shorter rename log lines.
# Example: RENAME_VERBOSE_SUGGESTIONS = True
RENAME_VERBOSE_SUGGESTIONS = True

# PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED
# True : Generate rename suggestions even if RENAME_IMAGES=False.
# False: Skip rename preview when RENAME_IMAGES=False.
# Example: PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED = True
PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED = True

# CONFIRM_EXECUTION
# True : Enable execution gate check for live operations.
# False: Live operations are blocked.
# Example: CONFIRM_EXECUTION = True
CONFIRM_EXECUTION = False

# CONFIRM_TEXT
# What: User confirmation string for live operations.
# Must exactly match REQUIRED_CONFIRM_TEXT.
# Example: CONFIRM_TEXT = "YES_DELETE_AND_RENAME"
CONFIRM_TEXT = ""

# REQUIRED_CONFIRM_TEXT
# What: Required phrase that must match CONFIRM_TEXT for execution.
# Example: REQUIRED_CONFIRM_TEXT = "YES_DELETE_AND_RENAME"
REQUIRED_CONFIRM_TEXT = "YES_DELETE_AND_RENAME"

# FOLLOW_SYMLINKS
# True : Recursive scanner follows symlinks.
# False: Symlinks are not followed.
# Example: FOLLOW_SYMLINKS = False
FOLLOW_SYMLINKS = False

# EXCLUDE_DIR_NAMES
# What: Directory names excluded from recursive scanning.
# Example: {".git", "_duplicate_cleaner_reports"}
EXCLUDE_DIR_NAMES = {
    ".git", ".idea", "__pycache__", "$RECYCLE.BIN",
    "_duplicate_cleaner_reports", "_duplicate_cleaner_quarantine",
}

# IMAGE_EXTENSIONS
# What: Extensions treated as images for image-only phases.
# Example: {".jpg", ".png", ".webp"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff", ".heic", ".heif", ".bmp"}

# INCLUDE_EXTENSIONS
# What: Optional extension filter for exact duplicate scan.
# Empty set => include all file extensions.
# Example: INCLUDE_EXTENSIONS = {".jpg", ".mp4"}
INCLUDE_EXTENSIONS = set()

print(f"ROOT_DIR = {ROOT_DIR}")
print(f"DRY_RUN = {DRY_RUN}, DELETE_DUPLICATES = {DELETE_DUPLICATES}, RENAME_IMAGES = {RENAME_IMAGES}")
print(f"FIND_SIMILAR_IMAGES = {FIND_SIMILAR_IMAGES}, FIND_AI_VISUAL_GROUPS = {FIND_AI_VISUAL_GROUPS}, MOVE_AI_GROUPS_TO_QUARANTINE = {MOVE_AI_GROUPS_TO_QUARANTINE}")
print(f"AI_QUARANTINE_DIR = {AI_QUARANTINE_DIR}")
print(f"VERBOSE = {VERBOSE}, LOG_EVERY_FILES = {LOG_EVERY_FILES}, LOG_EVERY_BUCKETS = {LOG_EVERY_BUCKETS}")
print(f"SHOW_PREVIEW_DETAILS = {SHOW_PREVIEW_DETAILS}")
print(f"SIMILAR_HASH_THRESHOLD = {SIMILAR_HASH_THRESHOLD}, AI_SIMILARITY_THRESHOLD = {AI_SIMILARITY_THRESHOLD}, AI_SIMILARITY_PERCENT = {AI_SIMILARITY_PERCENT}, AI_COMPARE_ALL_PAIRS = {AI_COMPARE_ALL_PAIRS}")
print(f"AI_DEVICE = {AI_DEVICE}, AI_USE_FP16_ON_CUDA = {AI_USE_FP16_ON_CUDA}, AI_ENABLE_TF32 = {AI_ENABLE_TF32}")
print(
    f"RENAME_WITH_AI_LABEL = {RENAME_WITH_AI_LABEL}, AI_RENAME_MODEL_NAME = {AI_RENAME_MODEL_NAME}, AI_RENAME_MODEL_FALLBACK = {AI_RENAME_MODEL_FALLBACK}, AI_RENAME_MAX_WORDS = {AI_RENAME_MAX_WORDS}, AI_RENAME_MAX_CHARS = {AI_RENAME_MAX_CHARS}, AI_RENAME_DATE_STYLE = {AI_RENAME_DATE_STYLE}, AI_RENAME_INCLUDE_TIME = {AI_RENAME_INCLUDE_TIME}, AI_RENAME_USE_CAPTION_SENTENCE = {AI_RENAME_USE_CAPTION_SENTENCE}, AI_SUPPRESS_MODEL_LOAD_WARNINGS = {AI_SUPPRESS_MODEL_LOAD_WARNINGS}, RENAME_ALL_IMAGES_WITH_AI_LABEL = {RENAME_ALL_IMAGES_WITH_AI_LABEL}, RENAME_VERBOSE_SUGGESTIONS = {RENAME_VERBOSE_SUGGESTIONS}, PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED = {PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED}")
print(f"CONFIRM_EXECUTION = {CONFIRM_EXECUTION}, CONFIRM_TEXT set = {bool(CONFIRM_TEXT)}")


ROOT_DIR = C:\Users\Carina\OneDrive
DRY_RUN = True, DELETE_DUPLICATES = False, RENAME_IMAGES = True
FIND_SIMILAR_IMAGES = True, FIND_AI_VISUAL_GROUPS = True, MOVE_AI_GROUPS_TO_QUARANTINE = False
AI_QUARANTINE_DIR = C:\Users\Carina\OneDrive\_duplicate_cleaner_quarantine\ai_visual
VERBOSE = True, LOG_EVERY_FILES = 500, LOG_EVERY_BUCKETS = 50
SHOW_PREVIEW_DETAILS = True
SIMILAR_HASH_THRESHOLD = 6, AI_SIMILARITY_THRESHOLD = 0.93, AI_SIMILARITY_PERCENT = 92.0, AI_COMPARE_ALL_PAIRS = True
AI_DEVICE = auto, AI_USE_FP16_ON_CUDA = True, AI_ENABLE_TF32 = True
RENAME_WITH_AI_LABEL = True, AI_RENAME_MODEL_NAME = Salesforce/blip-image-captioning-large, AI_RENAME_MODEL_FALLBACK = Salesforce/blip-image-captioning-base, AI_RENAME_MAX_WORDS = 0, AI_RENAME_MAX_CHARS = 250, AI_RENAME_DATE_STYLE = english_text, AI_RENAME_INCLUDE_TIME = False, AI_RENAME_USE_CAPTION_SENTENCE = True, AI_SUPPRESS_MODEL_LOAD_WARNINGS = True, RENAME_ALL_IMAGES_WITH_AI_LABEL = False, RENAME_VERBOSE_SUGGESTIONS = True, PREVIEW_RE

In [3]:
def iter_files(root: Path, exclude_dir_names: set[str], follow_symlinks: bool = False) -> Iterable[Path]:
    """Yields all files under root recursively, excluding selected directories."""
    root = root.resolve()

    # os.walk yields (current directory, subdirectories, files)
    for dirpath, dirnames, filenames in os.walk(root, topdown=True, followlinks=follow_symlinks):
        # Filter directories early so they are never entered.
        dirnames[:] = [d for d in dirnames if d not in exclude_dir_names]

        base = Path(dirpath)
        for name in filenames:
            p = base / name
            if p.is_file():
                yield p


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Computes the SHA-256 hash of a file in chunks (memory-efficient)."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def find_duplicate_groups(root: Path, include_extensions: set[str] | None = None) -> list[dict]:
    """
    Finds duplicates in two stages:
    1) Group by file size (fast pre-filter)
    2) Within equal-size groups, compare by SHA-256
    """
    log(f"[SCAN] Starting duplicate scan under: {root}")

    size_buckets: Dict[int, List[Path]] = defaultdict(list)
    include_extensions = {e.lower() for e in (include_extensions or set())}

    scanned_files = 0
    skipped_by_extension = 0

    # Initial pass: group files by size
    for p in iter_files(root, EXCLUDE_DIR_NAMES, follow_symlinks=FOLLOW_SYMLINKS):
        scanned_files += 1
        if LOG_EVERY_FILES and scanned_files % LOG_EVERY_FILES == 0:
            log(f"[SCAN] Files scanned: {scanned_files:,}")

        if include_extensions and p.suffix.lower() not in include_extensions:
            skipped_by_extension += 1
            continue

        try:
            size_buckets[p.stat().st_size].append(p)
        except OSError:
            # A file may disappear or be locked while scanning
            continue

    log(
        f"[SCAN] Done: {scanned_files:,} files, "
        f"{len(size_buckets):,} size groups, {skipped_by_extension:,} skipped by extension"
    )

    duplicate_groups: list[dict] = []
    candidate_buckets = [(size, paths) for size, paths in size_buckets.items() if len(paths) > 1]
    log(f"[HASH] Starting hash comparison for {len(candidate_buckets):,} candidate groups")

    hashed_files = 0
    for bucket_idx, (size, paths) in enumerate(candidate_buckets, start=1):
        if LOG_EVERY_BUCKETS and bucket_idx % LOG_EVERY_BUCKETS == 0:
            log(f"[HASH] Group {bucket_idx:,}/{len(candidate_buckets):,} (size={size} bytes, files={len(paths)})")

        hash_buckets: Dict[str, List[Path]] = defaultdict(list)
        for p in paths:
            try:
                digest = sha256_file(p)
                hash_buckets[digest].append(p)
                hashed_files += 1
                if LOG_EVERY_FILES and hashed_files % LOG_EVERY_FILES == 0:
                    log(f"[HASH] Files hashed: {hashed_files:,}")
            except OSError:
                continue

        # Each hash group with >=2 files is a true duplicate set
        for digest, dup_paths in hash_buckets.items():
            if len(dup_paths) < 2:
                continue

            # Keep rule:
            # 1) Oldest modification first (smallest mtime)
            # 2) On ties, shorter path
            # 3) Then alphabetical
            sorted_paths = sorted(
                dup_paths,
                key=lambda x: (
                    x.stat().st_mtime if x.exists() else float("inf"),
                    len(str(x)),
                    str(x).lower(),
                ),
            )
            keep = sorted_paths[0]
            remove = sorted_paths[1:]

            duplicate_groups.append({
                "hash": digest,
                "size": size,
                "keep": keep,
                "remove": remove,
                "count": len(sorted_paths),
            })

    duplicate_files = sum(len(g["remove"]) for g in duplicate_groups)
    log(f"[DONE] Duplicate groups: {len(duplicate_groups):,}, to remove: {duplicate_files:,}")

    return duplicate_groups


def build_duplicate_plan_actions(groups: list[dict]) -> list[dict]:
    """Builds a detailed delete preview per file (no actual changes)."""
    actions = []
    for idx, g in enumerate(groups, start=1):
        for p in g["remove"]:
            actions.append({
                "group": idx,
                "action": "delete",
                "path": str(p),
                "hash": g["hash"],
                "size": g["size"],
                "kept": str(g["keep"]),
                "status": "planned",
                "error": "",
            })
    return actions


def delete_duplicates(groups: list[dict], dry_run: bool = True) -> list[dict]:
    """Deletes all files marked as duplicates (except the kept file)."""
    total = sum(len(g["remove"]) for g in groups)
    mode = "DRY-RUN" if dry_run else "LIVE"
    log(f"[DELETE] Starting ({mode}), planned scope: {total:,} files")

    actions = []
    processed = 0

    for g in groups:
        for p in g["remove"]:
            processed += 1
            action = {
                "action": "delete",
                "path": str(p),
                "hash": g["hash"],
                "size": g["size"],
                "kept": str(g["keep"]),
                "status": "planned" if dry_run else "deleted",
                "error": "",
            }

            if dry_run:
                log(f"[PLAN DELETE] {p}")
            else:
                log(f"[DELETE] {p}")
                try:
                    p.unlink()  # Actually remove the file
                except OSError as e:
                    action["status"] = "error"
                    action["error"] = str(e)
                    log(f"[DELETE ERROR] {p} -> {e}")

            if LOG_EVERY_FILES and processed % LOG_EVERY_FILES == 0:
                log(f"[DELETE] Progress: {processed:,}/{total:,}")

            actions.append(action)

    log(f"[DELETE] Done, actions: {len(actions):,}")
    return actions


In [4]:
# Formats that typically contain an EXIF date
DATE_PATTERNS = [
    "%Y:%m:%d %H:%M:%S",
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%d",
]

# Typical cryptic filenames: IMG_20230213_..., PXL_..., DSC..., UUIDs, numeric-only names, etc.
CRYPTIC_NAME_RE = re.compile(
    r"^(?:"
    r"(?:img|dsc|pxl|mvimg|wp_|screenshot|photo)[-_ ]?\d.*"
    r"|\d{6,}"
    r"|[a-f0-9]{16,}"
    r"|[a-z0-9_-]{12,}"
    r")$",
    re.IGNORECASE,
)

# Already meaningful date-based names should not be renamed again
DATE_PREFIX_RE = re.compile(r"^\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}$")


def parse_exif_datetime(raw: str) -> Optional[datetime]:
    """Parses EXIF date text into a datetime object."""
    raw = (raw or "").strip()
    if not raw:
        return None

    for fmt in DATE_PATTERNS:
        try:
            return datetime.strptime(raw, fmt)
        except ValueError:
            pass

    return None


def get_image_datetime(path: Path) -> Optional[datetime]:
    """
    Prefer reading the capture date from EXIF:
    - DateTimeOriginal (36867)
    - DateTimeDigitized (36868)
    - DateTime (306)
    """
    if Image is None:
        return None

    try:
        with Image.open(path) as img:
            exif = getattr(img, "getexif", lambda: None)()
            if not exif:
                return None

            for tag_id in (36867, 36868, 306):
                raw = exif.get(tag_id)
                if raw:
                    dt = parse_exif_datetime(str(raw))
                    if dt:
                        return dt
    except Exception:
        # Corrupted file / unknown format -> simply skip
        return None

    return None


def is_cryptic_basename(stem: str) -> bool:
    """Heuristic to decide whether a filename looks cryptic and should be renamed."""
    s = stem.strip()
    if not s:
        return False

    # Already in target date format -> not cryptic
    if DATE_PREFIX_RE.match(s):
        return False

    # Match against known cryptic patterns
    if CRYPTIC_NAME_RE.match(s):
        return True

    # Additional rule: many digits, few letters
    letters = sum(ch.isalpha() for ch in s)
    digits = sum(ch.isdigit() for ch in s)
    if len(s) >= 8 and digits >= letters * 2:
        return True

    return False


def unique_path(target: Path) -> Path:
    """Create a unique target path using _1, _2, ... if needed."""
    if not target.exists():
        return target

    stem, suffix = target.stem, target.suffix
    i = 1
    while True:
        candidate = target.with_name(f"{stem}_{i}{suffix}")
        if not candidate.exists():
            return candidate
        i += 1


_AI_CAPTION_PIPELINE = None
_AI_CAPTION_PIPELINE_MODEL = ""
_AI_CAPTION_PIPELINE_FAILED = False
_AI_CAPTION_PIPELINE_TASK = ""

_AI_CAPTION_STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "at", "with", "for", "from",
    "is", "are", "was", "were", "this", "that", "these", "those", "it", "its", "by", "as",
    "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
}


def _resolve_ai_device(torch_mod):
    """Resolve preferred runtime device for AI workloads."""
    pref = str(globals().get("AI_DEVICE", "auto")).strip().lower()
    if pref not in {"auto", "cuda", "cpu"}:
        pref = "auto"

    cuda_ok = bool(getattr(torch_mod, "cuda", None) and torch_mod.cuda.is_available())

    # Hard compatibility check: avoid runtime crash on unsupported GPU arch
    # (e.g., sm_120 GPU with torch build that only contains kernels up to sm_90).
    if cuda_ok:
        try:
            major, minor = torch_mod.cuda.get_device_capability(0)
            device_sm = f"sm_{major}{minor}"
            arch_list = set(torch_mod.cuda.get_arch_list() or [])
            if arch_list and device_sm not in arch_list:
                log(
                    f"[AI] CUDA device {device_sm} is not supported by this PyTorch build "
                    f"(supported: {', '.join(sorted(arch_list))}). Falling back to CPU."
                )
                cuda_ok = False
        except Exception:
            pass

    if pref == "cpu":
        return "cpu", -1

    if pref == "cuda":
        if cuda_ok:
            return "cuda", 0
        log("[AI] AI_DEVICE='cuda' requested, but CUDA is unavailable/unsupported. Falling back to CPU.")
        return "cpu", -1

    # auto
    return ("cuda", 0) if cuda_ok else ("cpu", -1)


def _use_fp16_for_device(device: str) -> bool:
    return str(device).startswith("cuda") and bool(globals().get("AI_USE_FP16_ON_CUDA", True))


def _configure_ai_torch_backends(torch_mod, device: str) -> None:
    """Enable optional CUDA backend optimizations."""
    if not str(device).startswith("cuda"):
        return

    try:
        if bool(globals().get("AI_ENABLE_TF32", True)):
            torch_mod.backends.cuda.matmul.allow_tf32 = True
            torch_mod.backends.cudnn.allow_tf32 = True
        torch_mod.backends.cudnn.benchmark = True
    except Exception:
        pass


def _load_image_caption_pipeline(model_name: str):
    """Load and cache an image caption backend with robust version fallback."""
    global _AI_CAPTION_PIPELINE, _AI_CAPTION_PIPELINE_MODEL, _AI_CAPTION_PIPELINE_FAILED, _AI_CAPTION_PIPELINE_TASK

    if _AI_CAPTION_PIPELINE is not None and _AI_CAPTION_PIPELINE_MODEL == model_name:
        return _AI_CAPTION_PIPELINE

    if _AI_CAPTION_PIPELINE_FAILED and _AI_CAPTION_PIPELINE_MODEL == model_name:
        return None

    try:
        import torch
    except Exception:
        log("[RENAME-AI] Missing packages. Install with: pip install torch transformers")
        _AI_CAPTION_PIPELINE_FAILED = True
        _AI_CAPTION_PIPELINE_MODEL = model_name
        _AI_CAPTION_PIPELINE_TASK = ""
        return None

    device, device_id = _resolve_ai_device(torch)
    _configure_ai_torch_backends(torch, device)
    use_fp16 = _use_fp16_for_device(device)
    dtype_label = "fp16" if use_fp16 else "fp32"
    errors = []

    # Prefer direct BLIP inference for BLIP caption models.
    # This avoids pipeline task/API incompatibilities across transformers versions.
    if "blip-image-captioning" in model_name.lower():
        try:
            from transformers import BlipProcessor, BlipForConditionalGeneration
            import contextlib
            import io
            import warnings

            suppress_model_logs = bool(globals().get("AI_SUPPRESS_MODEL_LOAD_WARNINGS", True))
            model_kwargs = {}
            if use_fp16:
                model_kwargs["torch_dtype"] = torch.float16

            if suppress_model_logs:
                with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    warnings.filterwarnings("ignore", message=r".*tied weights mapping.*")
                    warnings.filterwarnings("ignore", message=r".*UNEXPECTED.*")
                    processor = BlipProcessor.from_pretrained(model_name)
                    model = BlipForConditionalGeneration.from_pretrained(model_name, **model_kwargs)
            else:
                processor = BlipProcessor.from_pretrained(model_name)
                model = BlipForConditionalGeneration.from_pretrained(model_name, **model_kwargs)

            model = model.to(device)
            if use_fp16 and hasattr(model, "half"):
                model = model.half()
            model.eval()

            _AI_CAPTION_PIPELINE = {
                "backend": "direct_blip",
                "processor": processor,
                "model": model,
                "device": device,
                "use_fp16": use_fp16,
            }
            _AI_CAPTION_PIPELINE_MODEL = model_name
            _AI_CAPTION_PIPELINE_FAILED = False
            _AI_CAPTION_PIPELINE_TASK = "direct_blip"
            log(f"[RENAME-AI] Caption model loaded: {model_name} on {device} (backend=direct_blip, dtype={dtype_label})")
            return _AI_CAPTION_PIPELINE
        except Exception as e:
            msg = str(e).replace("\n", " ").strip()
            if len(msg) > 260:
                msg = msg[:260] + "..."
            errors.append(f"direct_blip: {msg}")

    # Pipeline fallback for non-BLIP models or if direct BLIP load failed.
    try:
        from transformers import pipeline

        tasks_to_try = ["image-text-to-text", "image-to-text"]
        for task_name in tasks_to_try:
            try:
                pipe_kwargs = {}
                if use_fp16:
                    pipe_kwargs["model_kwargs"] = {"torch_dtype": torch.float16}
                pipe = pipeline(task_name, model=model_name, device=device_id, **pipe_kwargs)
                _AI_CAPTION_PIPELINE = pipe
                _AI_CAPTION_PIPELINE_MODEL = model_name
                _AI_CAPTION_PIPELINE_FAILED = False
                _AI_CAPTION_PIPELINE_TASK = task_name
                log(f"[RENAME-AI] Caption model loaded: {model_name} on {device} (task={task_name}, dtype={dtype_label})")
                return _AI_CAPTION_PIPELINE
            except Exception as e:
                msg = str(e).replace("\n", " ").strip()
                if len(msg) > 260:
                    msg = msg[:260] + "..."
                errors.append(f"{task_name}: {msg}")
    except Exception as e:
        msg = str(e).replace("\n", " ").strip()
        if len(msg) > 260:
            msg = msg[:260] + "..."
        errors.append(f"pipeline_import: {msg}")

    log(f"[RENAME-AI] Failed to load caption model ({model_name}) on {device}")
    for err in errors:
        log(f"[RENAME-AI]   {err}")

    _AI_CAPTION_PIPELINE = None
    _AI_CAPTION_PIPELINE_MODEL = model_name
    _AI_CAPTION_PIPELINE_FAILED = True
    _AI_CAPTION_PIPELINE_TASK = ""
    return None


def _caption_to_short_label(caption: str, max_words: int = 8, max_chars: int = 120) -> str:
    """
    Build an ASCII-safe filename label from caption text.

    - max_words > 0: CamelCase label up to max_words tokens.
    - max_words == 0: use full mini-sentence style with underscores.
    """
    max_words = int(max_words)
    max_chars = max(16, int(max_chars))

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", (caption or "").strip().lower())
    raw_tokens = [t for t in text.split() if len(t) > 0]

    if max_words == 0:
        words = [t for t in raw_tokens if len(t) > 1]
        label = "_".join(words)
        label = re.sub(r"_+", "_", label).strip("_")
        if len(label) > max_chars:
            label = label[:max_chars].rstrip("_")
        return label

    if max_words < 0:
        max_words = 1

    tokens = [t for t in raw_tokens if len(t) > 1 and t not in _AI_CAPTION_STOPWORDS]
    dedup = []
    seen = set()
    for tok in tokens:
        if tok in seen:
            continue
        seen.add(tok)
        dedup.append(tok)
        if len(dedup) >= max_words:
            break

    label = "".join(t.capitalize() for t in dedup)
    if len(label) > max_chars:
        label = label[:max_chars]
    return label


def _describe_image_for_filename(path: Path, caption_pipeline, max_words: int = 8, max_chars: int = 120) -> tuple[str, str]:
    """Return (short_label, raw_caption) for an image path."""
    if caption_pipeline is None:
        return "", ""

    caption = ""

    # Direct BLIP backend path
    if isinstance(caption_pipeline, dict) and caption_pipeline.get("backend") == "direct_blip":
        try:
            import contextlib
            import torch

            processor = caption_pipeline["processor"]
            model = caption_pipeline["model"]
            device = caption_pipeline.get("device", "cpu")
            use_fp16 = bool(caption_pipeline.get("use_fp16", False))
            is_cuda = str(device).startswith("cuda")

            with Image.open(path) as im:
                im = im.convert("RGB")
                inputs = processor(images=im, return_tensors="pt")

            inputs = {k: v.to(device) for k, v in inputs.items()}
            if is_cuda and use_fp16 and "pixel_values" in inputs:
                inputs["pixel_values"] = inputs["pixel_values"].to(dtype=torch.float16)

            autocast_ctx = (
                torch.autocast(device_type="cuda", dtype=torch.float16)
                if (is_cuda and use_fp16 and hasattr(torch, "autocast"))
                else contextlib.nullcontext()
            )
            with torch.inference_mode():
                with autocast_ctx:
                    generated = model.generate(**inputs, max_new_tokens=32)
            caption = processor.decode(generated[0], skip_special_tokens=True).strip()
        except Exception:
            return "", ""

        label = _caption_to_short_label(caption, max_words=max_words, max_chars=max_chars)
        return label, caption

    # Pipeline backend path
    try:
        out = caption_pipeline(str(path))
    except Exception:
        try:
            with Image.open(path) as im:
                out = caption_pipeline(im.convert("RGB"))
        except Exception:
            return "", ""

    if isinstance(out, list) and out:
        first = out[0]
        if isinstance(first, dict):
            caption = str(
                first.get("generated_text")
                or first.get("text")
                or first.get("caption")
                or first.get("answer")
                or ""
            ).strip()
        else:
            caption = str(first).strip()
    else:
        caption = str(out).strip()

    label = _caption_to_short_label(caption, max_words=max_words, max_chars=max_chars)
    return label, caption


def _format_rename_date_value(dt: datetime, style: str = "english_text", include_time: bool = False) -> str:
    """Format date prefix for renamed files."""
    style_norm = (style or "english_text").strip().lower()
    if style_norm == "iso":
        base = dt.strftime("%Y-%m-%d")
    else:
        # Example: 13. Feb.2026
        base = dt.strftime("%d. %b.%Y")

    if include_time:
        base = f"{base}_{dt.strftime('%H-%M-%S')}"
    return base


def _caption_to_sentence_filename(caption: str, max_chars: int = 120) -> str:
    """Convert AI caption text into a safe, readable filename sentence."""
    max_chars = max(16, int(max_chars))
    text = (caption or "").replace("\r", " ").replace("\n", " ").strip()
    if not text:
        return ""

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r'[<>:"/\\|?*]', "", text)
    text = text.strip(" ._-")

    if text:
        text = text[0].upper() + text[1:]

    if len(text) > max_chars:
        text = text[:max_chars].rstrip(" ._-")

    return text


def rename_cryptic_images(
        root: Path,
        dry_run: bool = True,
        ai_describe: bool = False,
        ai_model_name: str = "Salesforce/blip-image-captioning-large",
        ai_model_fallback: str = "Salesforce/blip-image-captioning-base",
        ai_max_words: int = 8,
        ai_max_chars: int = 120,
        enrich_date_only_names: bool = True,
        rename_all_images: bool = False,
        verbose_suggestions: bool = True,
        rename_date_style: str = "english_text",
        rename_include_time: bool = False,
        rename_caption_as_sentence: bool = True,
) -> list[dict]:
    """Rename image files with date prefix, optionally enriched by AI caption text."""
    mode = "DRY-RUN" if dry_run else "LIVE"
    log(f"[RENAME] Start ({mode})")

    actions: list[dict] = []
    scanned_images = 0
    rename_candidates = 0
    ai_labels_added = 0

    caption_pipeline = None
    active_caption_model = ""
    if ai_describe:
        candidate_models = [ai_model_name]
        if ai_model_fallback and ai_model_fallback != ai_model_name:
            candidate_models.append(ai_model_fallback)

        for m in candidate_models:
            caption_pipeline = _load_image_caption_pipeline(m)
            if caption_pipeline is not None:
                active_caption_model = m
                break

        if caption_pipeline is None:
            log("[RENAME-AI] AI label generation unavailable. Continuing with date-only names.")

    for p in iter_files(root, EXCLUDE_DIR_NAMES, follow_symlinks=FOLLOW_SYMLINKS):
        ext = p.suffix.lower()
        if ext not in IMAGE_EXTENSIONS:
            continue

        scanned_images += 1
        if LOG_EVERY_FILES and scanned_images % LOG_EVERY_FILES == 0:
            log(f"[RENAME] Images checked: {scanned_images:,}")

        stem = p.stem
        is_cryptic = is_cryptic_basename(stem)
        is_date_only = bool(DATE_PREFIX_RE.match(stem))

        should_rename = rename_all_images or is_cryptic or (ai_describe and enrich_date_only_names and is_date_only)
        if not should_rename:
            continue

        rename_candidates += 1

        # Prefer EXIF capture date; fall back to file mtime.
        dt = get_image_datetime(p)
        if dt is None:
            try:
                dt = datetime.fromtimestamp(p.stat().st_mtime)
            except OSError:
                continue

        date_prefix = _format_rename_date_value(dt, style=rename_date_style, include_time=rename_include_time)
        new_stem = date_prefix
        ai_label = ""
        ai_caption = ""

        if ai_describe and caption_pipeline is not None:
            ai_label, ai_caption = _describe_image_for_filename(
                p,
                caption_pipeline,
                max_words=ai_max_words,
                max_chars=ai_max_chars,
            )
            if ai_label:
                if rename_caption_as_sentence and ai_caption:
                    sentence = _caption_to_sentence_filename(ai_caption, max_chars=ai_max_chars)
                    new_stem = f"{date_prefix}_{sentence}" if sentence else f"{date_prefix}_{ai_label}"
                else:
                    new_stem = f"{date_prefix}_{ai_label}"
                ai_labels_added += 1

        if stem == new_stem:
            continue

        target = unique_path(p.with_name(new_stem + ext))

        action = {
            "action": "rename",
            "from": str(p),
            "to": str(target),
            "status": "planned" if dry_run else "renamed",
            "error": "",
            "ai_label": ai_label,
            "ai_caption": ai_caption,
            "ai_model": active_caption_model if ai_describe else "",
        }

        ai_info = f" | ai_label={ai_label} | ai_caption={ai_caption}" if ai_label or ai_caption else ""
        if dry_run:
            if verbose_suggestions:
                log(f"[PLAN RENAME] found={p} | current_name={p.name} | proposed_name={target.name} | proposed_path={target}{ai_info}")
            else:
                log(f"[PLAN RENAME] {p.name} -> {target.name}")
        else:
            if verbose_suggestions:
                log(f"[RENAME] from={p} | to={target}{ai_info}")
            else:
                log(f"[RENAME] {p.name} -> {target.name}")
            try:
                p.rename(target)
            except OSError as e:
                action["status"] = "error"
                action["error"] = str(e)
                log(f"[RENAME ERROR] {p} -> {e}")

        actions.append(action)

    log(
        f"[RENAME] Done: checked_images={scanned_images:,}, "
        f"candidates={rename_candidates:,}, actions={len(actions):,}, ai_labels={ai_labels_added:,}"
    )
    return actions


def dhash64_from_path(path: Path) -> Optional[int]:
    """Perceptual hash (dHash, 64-bit) for similarity comparison."""
    if Image is None:
        return None

    try:
        with Image.open(path) as img:
            gray = img.convert("L").resize((9, 8))
            pixels = list(gray.getdata())

        digest = 0
        bit = 0
        for y in range(8):
            row = y * 9
            for x in range(8):
                left_px = pixels[row + x]
                right_px = pixels[row + x + 1]
                if left_px > right_px:
                    digest |= (1 << bit)
                bit += 1

        return digest
    except Exception:
        return None


def hamming_distance_64(a: int, b: int) -> int:
    """Number of differing bits between two 64-bit hashes."""
    return (a ^ b).bit_count()


def find_similar_image_groups(
        root: Path,
        threshold: int = 6,
        neighbor_window: int = 20,
        min_file_size_bytes: int = 0,
) -> dict:
    """
    Finds very similar images (near-duplicates) via dHash + Hamming distance.

    Performance idea:
    - compare only images with the same resolution
    - within each resolution, compare only with the next N images (by time)
    """
    if Image is None:
        log("[SIMILAR] Pillow not available -> similarity analysis skipped")
        return {"groups": [], "actions": [], "pairs": []}

    threshold = max(0, min(64, int(threshold)))
    neighbor_window = max(1, int(neighbor_window))

    log(
        f"[SIMILAR] Starting (threshold={threshold}, window={neighbor_window}, min_size={min_file_size_bytes} bytes)"
    )

    signatures = []
    scanned_images = 0
    skipped_small = 0

    for p in iter_files(root, EXCLUDE_DIR_NAMES, follow_symlinks=FOLLOW_SYMLINKS):
        ext = p.suffix.lower()
        if ext not in IMAGE_EXTENSIONS:
            continue

        scanned_images += 1
        if LOG_EVERY_FILES and scanned_images % LOG_EVERY_FILES == 0:
            log(f"[SIMILAR] Images checked: {scanned_images:,}")

        try:
            stat = p.stat()
        except OSError:
            continue

        if stat.st_size < min_file_size_bytes:
            skipped_small += 1
            continue

        digest = dhash64_from_path(p)
        if digest is None:
            continue

        dt = get_image_datetime(p)
        ts = dt.timestamp() if dt else stat.st_mtime

        signatures.append(
            {
                "path": p,
                "hash": digest,
                "width": None,
                "height": None,
                "timestamp": ts,
                "mtime": stat.st_mtime,
            }
        )

    # Read image dimensions separately for resolution bucketing
    prepared = []
    for item in signatures:
        p = item["path"]
        try:
            with Image.open(p) as img:
                w, h = img.size
        except Exception:
            continue

        item["width"] = w
        item["height"] = h
        prepared.append(item)

    signatures = prepared
    log(
        f"[SIMILAR] Signaturen erstellt: {len(signatures):,} "
        f"(gescannt={scanned_images:,}, zu_klein={skipped_small:,})"
    )

    buckets: Dict[tuple[int, int], List[dict]] = defaultdict(list)
    for s in signatures:
        buckets[(s["width"], s["height"])].append(s)

    pairs = []
    bucket_items = list(buckets.items())
    for b_idx, ((w, h), items) in enumerate(bucket_items, start=1):
        if len(items) < 2:
            continue

        if LOG_EVERY_BUCKETS and b_idx % LOG_EVERY_BUCKETS == 0:
            log(f"[SIMILAR] Bucket {b_idx:,}/{len(bucket_items):,}: {w}x{h}, files={len(items):,}")

        items_sorted = sorted(items, key=lambda x: (x["timestamp"], str(x["path"]).lower()))

        for i, a in enumerate(items_sorted):
            end = min(len(items_sorted), i + 1 + neighbor_window)
            for j in range(i + 1, end):
                b = items_sorted[j]
                dist = hamming_distance_64(a["hash"], b["hash"])
                if dist <= threshold:
                    pairs.append(
                        {
                            "path_a": str(a["path"]),
                            "path_b": str(b["path"]),
                            "distance": dist,
                            "width": w,
                            "height": h,
                        }
                    )

    if not pairs:
        log("[SIMILAR] No similar images found")
        return {"groups": [], "actions": [], "pairs": []}

    # Union-Find for group construction from pairs
    parent: Dict[str, str] = {}

    def uf_find(x: str) -> str:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def uf_union(a: str, b: str) -> None:
        ra = uf_find(a)
        rb = uf_find(b)
        if ra != rb:
            parent[rb] = ra

    for pair in pairs:
        a = pair["path_a"]
        b = pair["path_b"]
        if a not in parent:
            parent[a] = a
        if b not in parent:
            parent[b] = b
        uf_union(a, b)

    comps: Dict[str, List[str]] = defaultdict(list)
    for node in parent:
        comps[uf_find(node)].append(node)

    groups = []
    actions = []

    def mtime_safe(path_str: str) -> float:
        try:
            return Path(path_str).stat().st_mtime
        except OSError:
            return float("inf")

    for gid, paths in enumerate(sorted(comps.values(), key=lambda x: len(x), reverse=True), start=1):
        if len(paths) < 2:
            continue

        sorted_paths = sorted(paths, key=lambda s: (mtime_safe(s), len(s), s.lower()))
        keep = sorted_paths[0]
        similar = sorted_paths[1:]

        groups.append(
            {
                "group": gid,
                "keep": keep,
                "similar": similar,
                "count": len(sorted_paths),
            }
        )

        for p in similar:
            actions.append(
                {
                    "group": gid,
                    "action": "flag_similar",
                    "keep": keep,
                    "similar": p,
                    "status": "flagged",
                }
            )

    log(f"[SIMILAR] Done: groups={len(groups):,}, markierte files={len(actions):,}, pairs={len(pairs):,}")
    return {"groups": groups, "actions": actions, "pairs": pairs}


In [5]:
def log(message: str) -> None:
    """Small timestamp logger controlled by VERBOSE."""
    if VERBOSE:
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] {message}")


def should_execute_changes() -> bool:
    """Strict safety check before real file changes are allowed."""
    if DRY_RUN:
        print("Execution blocked: DRY_RUN=True")
        return False

    if not CONFIRM_EXECUTION:
        print("Execution blocked: CONFIRM_EXECUTION=False")
        return False

    if (CONFIRM_TEXT or "").strip() != REQUIRED_CONFIRM_TEXT:
        print("Execution blocked: CONFIRM_TEXT does not match REQUIRED_CONFIRM_TEXT.")
        print(f"Expected: {REQUIRED_CONFIRM_TEXT}")
        return False

    return True


def print_duplicate_plan(groups: list[dict], max_groups: Optional[int] = None) -> None:
    """Prints the comparison: KEEP file and duplicates to delete."""
    if not groups:
        print("No duplicates found.")
        return

    total_groups = len(groups)
    shown = groups
    if max_groups is not None and max_groups > 0:
        shown = groups[:max_groups]

    for idx, g in enumerate(shown, start=1):
        print(f"\n[Duplicate Group {idx}] Hash={g['hash'][:12]}... | size={g['size']} bytes | files={g['count']}")
        print(f"  KEEP: {g['keep']}")
        for p in g["remove"]:
            print(f"  DELETE : {p}")

    if len(shown) < total_groups:
        print(f"\n... {total_groups - len(shown)} more groups not printed (see CSV report).")


def print_rename_plan(actions: list[dict], max_rows: Optional[int] = None) -> None:
    """Prints old/new image name comparison for preview."""
    if not actions:
        print("No renames planned.")
        return

    shown = actions
    if max_rows is not None and max_rows > 0:
        shown = actions[:max_rows]

    for idx, a in enumerate(shown, start=1):
        print(f"[{idx}] FROM: {a['from']}")
        print(f"    TO  : {a['to']}")

    if len(shown) < len(actions):
        print(f"... {len(actions) - len(shown)} more renames not printed (see CSV report).")


def print_similar_plan(groups: list[dict], max_groups: Optional[int] = None) -> None:
    """Prints very similar images as groups: REFERENCE + SIMILAR."""
    if not groups:
        print("No similar image groups found.")
        return

    total_groups = len(groups)
    shown = groups
    if max_groups is not None and max_groups > 0:
        shown = groups[:max_groups]

    for g in shown:
        print(f"\n[Similarity Group {g['group']}] files={g['count']}")
        print(f"  REFERENCE: {g['keep']}")
        for p in g["similar"]:
            print(f"  SIMILAR: {p}")

    if len(shown) < total_groups:
        print(f"\n... {total_groups - len(shown)} more groups not printed (see CSV report).")


def save_csv(path: Path, rows: list[dict]) -> None:
    """Saves action lists as a CSV report."""
    if not rows:
        return

    fieldnames = sorted({k for row in rows for k in row.keys()})
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def summarize_duplicate_groups(groups: list[dict]) -> dict:
    """Computes a compact summary of found duplicates."""
    total_groups = len(groups)
    total_duplicate_files = sum(len(g["remove"]) for g in groups)
    total_duplicate_bytes = sum(g["size"] * len(g["remove"]) for g in groups)

    return {
        "groups": total_groups,
        "duplicate_files": total_duplicate_files,
        "duplicate_bytes": total_duplicate_bytes,
        "duplicate_gb": round(total_duplicate_bytes / (1024 ** 3), 3),
    }


def build_ai_quarantine_plan(ai_groups: list[dict], root_dir: Path, quarantine_dir: Path) -> list[dict]:
    """Plans moves: keep 1 file per AI group, move the rest to quarantine."""
    actions = []

    for g in ai_groups:
        keep = g.get("keep")
        for src_str in g.get("similar", []):
            src = Path(src_str)
            if not src.exists():
                actions.append({
                    "group": g.get("group"),
                    "action": "move_to_quarantine",
                    "keep": keep,
                    "from": str(src),
                    "to": "",
                    "status": "missing",
                    "error": "source_missing",
                })
                continue

            try:
                rel = src.relative_to(root_dir)
            except ValueError:
                rel = Path(src.name)

            target = quarantine_dir / rel
            target = unique_path(target)

            actions.append({
                "group": g.get("group"),
                "action": "move_to_quarantine",
                "keep": keep,
                "from": str(src),
                "to": str(target),
                "status": "planned",
                "error": "",
            })

    return actions


def print_ai_quarantine_plan(actions: list[dict], max_rows: Optional[int] = None) -> None:
    """Prints planned quarantine moves."""
    if not actions:
        print("No AI quarantine actions planned.")
        return

    shown = actions
    if max_rows is not None and max_rows > 0:
        shown = actions[:max_rows]

    for idx, a in enumerate(shown, start=1):
        print(f"[{idx}] GROUP {a.get('group')} | KEEP: {a.get('keep')}")
        print(f"    VERSCHIEBEN: {a.get('from')}")
        print(f"    NACH      : {a.get('to')}")

    if len(shown) < len(actions):
        print(f"... {len(actions) - len(shown)} more quarantine actions not printed (see CSV report).")


def execute_ai_quarantine(actions: list[dict], dry_run: bool = True) -> list[dict]:
    """Executes the planned quarantine moves."""
    mode = "DRY-RUN" if dry_run else "LIVE"
    log(f"[AI-QUAR] Starting ({mode}), planned scope: {len(actions):,} files")

    out = []
    for idx, a in enumerate(actions, start=1):
        row = dict(a)

        if row.get("status") == "missing":
            out.append(row)
            continue

        src = Path(row["from"])
        dst = Path(row["to"])

        if dry_run:
            log(f"[AI-QUAR PLAN] {src} -> {dst}")
            out.append(row)
            continue

        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            row["status"] = "moved"
            log(f"[AI-QUAR MOVE] {src} -> {dst}")
        except Exception as e:
            row["status"] = "error"
            row["error"] = str(e)
            log(f"[AI-QUAR ERROR] {src} -> {e}")

        if LOG_EVERY_FILES and idx % LOG_EVERY_FILES == 0:
            log(f"[AI-QUAR] Progress: {idx:,}/{len(actions):,}")

        out.append(row)

    log(f"[AI-QUAR] Done, actions: {len(out):,}")
    return out


In [6]:
# Documentation + validation helpers
# Run these helpers whenever you want explicit explanations and a quick config sanity check.

SETTINGS_REFERENCE = {
    "ROOT_DIR": {
        "what": "Root directory scanned recursively.",
        "example": "ROOT_DIR = Path('X:/Privat/OneDrive')",
    },
    "DRY_RUN": {
        "what": "Global safety switch. True means preview only.",
        "example": "DRY_RUN = True",
    },
    "DELETE_DUPLICATES": {"what": "Enable exact duplicate deletion phase.", "example": "DELETE_DUPLICATES = True"},
    "RENAME_IMAGES": {"what": "Enable rename phase execution path.", "example": "RENAME_IMAGES = True"},
    "FIND_SIMILAR_IMAGES": {"what": "Enable dHash near-duplicate detection.", "example": "FIND_SIMILAR_IMAGES = True"},
    "FIND_AI_VISUAL_GROUPS": {"what": "Enable CLIP visual grouping phase.", "example": "FIND_AI_VISUAL_GROUPS = True"},
    "MOVE_AI_GROUPS_TO_QUARANTINE": {"what": "Enable moving AI-similar files to quarantine.", "example": "MOVE_AI_GROUPS_TO_QUARANTINE = True"},
    "AI_QUARANTINE_DIR": {"what": "Destination for AI quarantine moves.", "example": "AI_QUARANTINE_DIR = ROOT_DIR / '_duplicate_cleaner_quarantine' / 'ai_visual'"},
    "VERBOSE": {"what": "Enable timestamp logs.", "example": "VERBOSE = True"},
    "LOG_EVERY_FILES": {"what": "Progress log interval for file loops.", "example": "LOG_EVERY_FILES = 500"},
    "LOG_EVERY_BUCKETS": {"what": "Progress log interval for bucket loops.", "example": "LOG_EVERY_BUCKETS = 50"},
    "SHOW_PREVIEW_DETAILS": {"what": "Print detailed preview lists.", "example": "SHOW_PREVIEW_DETAILS = True"},
    "PREVIEW_MAX_DUPLICATE_GROUPS": {"what": "Duplicate preview limit (0=all).", "example": "PREVIEW_MAX_DUPLICATE_GROUPS = 25"},
    "PREVIEW_MAX_RENAMES": {"what": "Rename preview limit (0=all).", "example": "PREVIEW_MAX_RENAMES = 100"},
    "PREVIEW_MAX_SIMILAR_GROUPS": {"what": "Near-duplicate preview limit (0=all).", "example": "PREVIEW_MAX_SIMILAR_GROUPS = 50"},
    "PREVIEW_MAX_AI_GROUPS": {"what": "AI-group preview limit (0=all).", "example": "PREVIEW_MAX_AI_GROUPS = 20"},
    "PREVIEW_MAX_AI_QUARANTINE": {"what": "Quarantine preview limit (0=all).", "example": "PREVIEW_MAX_AI_QUARANTINE = 100"},
    "SIMILAR_HASH_THRESHOLD": {"what": "Near-duplicate distance threshold (0..64, lower=stricter).", "example": "SIMILAR_HASH_THRESHOLD = 6"},
    "SIMILAR_NEIGHBOR_WINDOW": {"what": "Near-duplicate compare window per image.", "example": "SIMILAR_NEIGHBOR_WINDOW = 20"},
    "SIMILAR_MIN_FILE_SIZE_BYTES": {"what": "Minimum image size for near-duplicate phase.", "example": "SIMILAR_MIN_FILE_SIZE_BYTES = 20000"},
    "AI_MODEL_NAME": {"what": "CLIP model for AI visual grouping.", "example": "AI_MODEL_NAME = 'openai/clip-vit-base-patch32'"},
    "AI_SIMILARITY_THRESHOLD": {"what": "Fallback similarity threshold (0..1).", "example": "AI_SIMILARITY_THRESHOLD = 0.93"},
    "AI_SIMILARITY_PERCENT": {"what": "Preferred similarity threshold in percent.", "example": "AI_SIMILARITY_PERCENT = 92.0"},
    "AI_NEIGHBOR_WINDOW": {"what": "Neighbor compare window when not exhaustive.", "example": "AI_NEIGHBOR_WINDOW = 60"},
    "AI_MIN_FILE_SIZE_BYTES": {"what": "Minimum image size for AI grouping.", "example": "AI_MIN_FILE_SIZE_BYTES = 20000"},
    "AI_BATCH_SIZE": {"what": "Batch size for CLIP embedding extraction.", "example": "AI_BATCH_SIZE = 16"},
    "AI_DEVICE": {"what": "AI runtime device preference (auto/cuda/cpu).", "example": "AI_DEVICE = 'auto'"},
    "AI_USE_FP16_ON_CUDA": {"what": "Use float16 on CUDA for speed/VRAM efficiency.", "example": "AI_USE_FP16_ON_CUDA = True"},
    "AI_ENABLE_TF32": {"what": "Enable TF32 acceleration on NVIDIA GPUs.", "example": "AI_ENABLE_TF32 = True"},
    "AI_COMPARE_ALL_PAIRS": {"what": "Exhaustive pair comparison mode (can be very slow).", "example": "AI_COMPARE_ALL_PAIRS = True"},
    "AI_VERBOSE": {"what": "Verbose logs for AI grouping phase.", "example": "AI_VERBOSE = True"},
    "AI_VERBOSE_PAIR_LOG_LIMIT": {"what": "Max detailed matched-pair logs.", "example": "AI_VERBOSE_PAIR_LOG_LIMIT = 25"},
    "RENAME_WITH_AI_LABEL": {"what": "Add AI semantic label to timestamp rename target.", "example": "RENAME_WITH_AI_LABEL = True"},
    "AI_RENAME_MODEL_NAME": {"what": "Primary image-caption model for labels.", "example": "AI_RENAME_MODEL_NAME = 'Salesforce/blip-image-captioning-large'"},
    "AI_RENAME_MODEL_FALLBACK": {"what": "Fallback caption model if primary fails.", "example": "AI_RENAME_MODEL_FALLBACK = 'Salesforce/blip-image-captioning-base'"},
    "AI_RENAME_MAX_WORDS": {"what": "Max words in label; 0 means full mini-sentence label.", "example": "AI_RENAME_MAX_WORDS = 0"},
    "AI_RENAME_MAX_CHARS": {"what": "Max output label character length.", "example": "AI_RENAME_MAX_CHARS = 120"},
    "AI_RENAME_DATE_STYLE": {"what": "Date prefix style for renamed images (english_text or iso).", "example": "AI_RENAME_DATE_STYLE = 'english_text'"},
    "AI_RENAME_INCLUDE_TIME": {"what": "Append _HH-MM-SS after date prefix.", "example": "AI_RENAME_INCLUDE_TIME = False"},
    "AI_RENAME_USE_CAPTION_SENTENCE": {"what": "Use readable AI caption sentence in filename instead of compact token label.", "example": "AI_RENAME_USE_CAPTION_SENTENCE = True"},
    "AI_SUPPRESS_MODEL_LOAD_WARNINGS": {"what": "Suppress noisy model-load warnings/reports for caption models.", "example": "AI_SUPPRESS_MODEL_LOAD_WARNINGS = True"},
    "AI_ENRICH_DATE_ONLY_NAMES": {"what": "Allow enriching date-only names with AI label.", "example": "AI_ENRICH_DATE_ONLY_NAMES = True"},
    "RENAME_ALL_IMAGES_WITH_AI_LABEL": {"what": "Rename all images, not only cryptic/date-only ones.", "example": "RENAME_ALL_IMAGES_WITH_AI_LABEL = True"},
    "RENAME_VERBOSE_SUGGESTIONS": {"what": "Verbose rename suggestion logs with full paths and captions.", "example": "RENAME_VERBOSE_SUGGESTIONS = True"},
    "PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED": {"what": "Generate rename suggestions even when RENAME_IMAGES=False.", "example": "PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED = True"},
    "CONFIRM_EXECUTION": {"what": "Execution gate for live actions.", "example": "CONFIRM_EXECUTION = True"},
    "CONFIRM_TEXT": {"what": "Must match REQUIRED_CONFIRM_TEXT for live actions.", "example": "CONFIRM_TEXT = 'YES_DELETE_AND_RENAME'"},
    "REQUIRED_CONFIRM_TEXT": {"what": "Expected confirmation phrase.", "example": "REQUIRED_CONFIRM_TEXT = 'YES_DELETE_AND_RENAME'"},
    "FOLLOW_SYMLINKS": {"what": "Whether recursive scan follows symlinks.", "example": "FOLLOW_SYMLINKS = False"},
    "EXCLUDE_DIR_NAMES": {"what": "Directory names excluded from scan.", "example": "EXCLUDE_DIR_NAMES = {'.git', '_duplicate_cleaner_reports'}"},
    "IMAGE_EXTENSIONS": {"what": "Image extensions considered for image phases.", "example": "IMAGE_EXTENSIONS = {'.jpg', '.png'}"},
    "INCLUDE_EXTENSIONS": {"what": "Optional filter for exact duplicate scan.", "example": "INCLUDE_EXTENSIONS = {'.jpg', '.png'}"},
}

METHOD_REFERENCE = {
    "iter_files": "Recursive file iterator with excluded directories.",
    "sha256_file": "Chunked SHA-256 hash for exact duplicate verification.",
    "find_duplicate_groups": "Two-stage exact duplicate finder (size bucket + hash).",
    "build_duplicate_plan_actions": "Flattens duplicate groups into delete plan rows.",
    "delete_duplicates": "Executes deletion for planned duplicate files.",
    "parse_exif_datetime": "Parses EXIF datetime strings with fallback formats.",
    "get_image_datetime": "Reads EXIF capture date (DateTimeOriginal/DateTimeDigitized/DateTime).",
    "is_cryptic_basename": "Heuristic to detect noisy/non-descriptive filenames.",
    "unique_path": "Generates collision-safe path by adding _1, _2, ...",
    "_load_image_caption_pipeline": "Loads and caches local image-to-text model pipeline.",
    "_caption_to_short_label": "Converts caption to filename-safe label (word-limited or full sentence).",
    "_describe_image_for_filename": "Runs caption model and returns (label, raw_caption).",
    "rename_cryptic_images": "Creates rename suggestions and optionally executes renames.",
    "dhash64_from_path": "Computes 64-bit perceptual hash for near-duplicate matching.",
    "hamming_distance_64": "Computes Hamming distance between two 64-bit hashes.",
    "find_similar_image_groups": "Near-duplicate grouping based on dHash distance + Union-Find.",
    "log": "Timestamp logger controlled by VERBOSE.",
    "should_execute_changes": "Hard execution gate validation.",
    "print_duplicate_plan": "Preview printer for duplicate keep/delete groups.",
    "print_rename_plan": "Preview printer for rename actions.",
    "print_similar_plan": "Preview printer for near-duplicate groups.",
    "save_csv": "Generic CSV writer for report rows.",
    "summarize_duplicate_groups": "Summary metrics for duplicate groups.",
    "build_ai_quarantine_plan": "Creates move plan from AI groups to quarantine directory.",
    "print_ai_quarantine_plan": "Preview printer for quarantine move actions.",
    "execute_ai_quarantine": "Executes planned quarantine moves.",
    "_ai_print_groups": "Preview printer for AI visual groups.",
    "_load_clip_model": "Loads CLIP model + processor on CPU/GPU.",
    "_resolve_similarity_threshold": "Normalizes similarity threshold from percent/float input.",
    "find_ai_visual_groups": "AI visual grouping via CLIP embeddings + pair search + Union-Find.",
}

METHOD_SUBBLOCKS = {
    "find_duplicate_groups": [
        "scan stage (size buckets)",
        "hash stage (sha256)",
        "group reduction (keep/remove)",
    ],
    "find_similar_image_groups": [
        "signature extraction (dHash)",
        "resolution bucketing",
        "windowed pair comparisons",
        "union-find group assembly",
    ],
    "find_ai_visual_groups": [
        "candidate scan + filtering",
        "batched CLIP embeddings",
        "pair similarity search (windowed or exhaustive)",
        "union-find group assembly",
    ],
    "rename_cryptic_images": [
        "candidate selection rules",
        "date derivation (EXIF -> mtime)",
        "optional AI caption label generation",
        "collision-safe target path creation",
        "dry-run preview or live rename",
    ],
}


def show_settings_reference(filter_text: Optional[str] = None) -> None:
    """Print explicit setting descriptions with examples."""
    needle = (filter_text or "").strip().lower()
    keys = sorted(SETTINGS_REFERENCE.keys())
    printed = 0

    for key in keys:
        item = SETTINGS_REFERENCE[key]
        line = f"{key}: {item['what']} | Example: {item['example']}"
        if needle and needle not in line.lower():
            continue
        print(line)
        printed += 1

    if printed == 0:
        print(f"No settings matched filter: {filter_text}")


def show_method_reference(filter_text: Optional[str] = None) -> None:
    """Print method descriptions and explicit sub-block notes where available."""
    needle = (filter_text or "").strip().lower()
    keys = sorted(METHOD_REFERENCE.keys())
    printed = 0

    for key in keys:
        desc = METHOD_REFERENCE[key]
        header = f"{key}: {desc}"
        if needle and needle not in header.lower() and needle not in " ".join(METHOD_SUBBLOCKS.get(key, [])).lower():
            continue
        print(header)
        for sub in METHOD_SUBBLOCKS.get(key, []):
            print(f"  - block: {sub}")
        printed += 1

    if printed == 0:
        print(f"No methods matched filter: {filter_text}")


def validate_configuration(print_ok: bool = True) -> list[str]:
    """Validate key setting ranges/dependencies and return warnings list."""
    warnings = []

    if not (0 <= SIMILAR_HASH_THRESHOLD <= 64):
        warnings.append("SIMILAR_HASH_THRESHOLD should be in [0, 64].")

    if AI_SIMILARITY_PERCENT is not None and not (0 <= float(AI_SIMILARITY_PERCENT) <= 100):
        warnings.append("AI_SIMILARITY_PERCENT should be in [0, 100].")

    if not (0 <= float(AI_SIMILARITY_THRESHOLD) <= 1.5):
        warnings.append("AI_SIMILARITY_THRESHOLD looks unusual. Expected around 0..1 (or percent-like >1).")

    if AI_BATCH_SIZE < 1:
        warnings.append("AI_BATCH_SIZE must be >= 1.")

    if str(AI_DEVICE).strip().lower() not in {"auto", "cuda", "cpu"}:
        warnings.append("AI_DEVICE should be one of: auto, cuda, cpu.")

    try:
        import torch as _torch
        if str(AI_DEVICE).strip().lower() == "cuda" and not _torch.cuda.is_available():
            warnings.append("AI_DEVICE='cuda' but torch.cuda.is_available() is False. Install CUDA-enabled torch build.")
    except Exception:
        pass

    if SIMILAR_NEIGHBOR_WINDOW < 1:
        warnings.append("SIMILAR_NEIGHBOR_WINDOW must be >= 1.")

    if AI_NEIGHBOR_WINDOW < 1:
        warnings.append("AI_NEIGHBOR_WINDOW must be >= 1.")

    if AI_RENAME_MAX_WORDS < 0:
        warnings.append("AI_RENAME_MAX_WORDS must be >= 0 (0 means full sentence label).")

    if AI_RENAME_MAX_CHARS < 16:
        warnings.append("AI_RENAME_MAX_CHARS should be >= 16 for readable labels.")

    if str(AI_RENAME_DATE_STYLE).strip().lower() not in {"english_text", "iso"}:
        warnings.append("AI_RENAME_DATE_STYLE should be one of: english_text, iso.")

    if DELETE_DUPLICATES and DRY_RUN:
        warnings.append("DELETE_DUPLICATES=True but DRY_RUN=True. No deletion will execute.")

    if RENAME_IMAGES and DRY_RUN:
        warnings.append("RENAME_IMAGES=True but DRY_RUN=True. Rename execution remains preview-only.")

    if CONFIRM_EXECUTION and (CONFIRM_TEXT or "").strip() != REQUIRED_CONFIRM_TEXT:
        warnings.append("CONFIRM_EXECUTION=True but CONFIRM_TEXT does not match REQUIRED_CONFIRM_TEXT.")

    missing_docs = [name for name in METHOD_REFERENCE.keys() if name not in globals()]
    if missing_docs:
        warnings.append(f"Some documented methods are not defined yet in current runtime: {', '.join(missing_docs[:8])}")

    if warnings:
        print("[CONFIG CHECK] Warnings:")
        for w in warnings:
            print(f"- {w}")
    elif print_ok:
        print("[CONFIG CHECK] OK: no obvious issues found.")

    return warnings


print("Documentation helpers ready:")
print("- show_settings_reference()")
print("- show_method_reference()")
print("- validate_configuration()")
_ = validate_configuration(print_ok=True)


Documentation helpers ready:
- show_settings_reference()
- show_method_reference()
- validate_configuration()
[CONFIG CHECK] Warnings:
- RENAME_IMAGES=True but DRY_RUN=True. Rename execution remains preview-only.
- Some documented methods are not defined yet in current runtime: _ai_print_groups, _load_clip_model, _resolve_similarity_threshold, find_ai_visual_groups


In [7]:
# 1) Find duplicates + generate a detailed preview
# This cell does not modify any files yet.
log("===== PHASE 1: SCAN DUPLICATES =====")
groups = find_duplicate_groups(ROOT_DIR, INCLUDE_EXTENSIONS)
summary = summarize_duplicate_groups(groups)

duplicate_plan_actions = build_duplicate_plan_actions(groups)

print("Duplicate summary:")
for k, v in summary.items():
    print(f"- {k}: {v}")
print(f"- planned deletion scope (files): {len(duplicate_plan_actions)}")

if SHOW_PREVIEW_DETAILS:
    limit = None if PREVIEW_MAX_DUPLICATE_GROUPS <= 0 else PREVIEW_MAX_DUPLICATE_GROUPS
    print_duplicate_plan(groups, max_groups=limit)
else:
    print("Detail preview disabled (SHOW_PREVIEW_DETAILS=False).")


[21:48:05] ===== PHASE 1: SCAN DUPLICATES =====
[21:48:05] [SCAN] Starting duplicate scan under: C:\Users\Carina\OneDrive
[21:48:05] [SCAN] Files scanned: 500
[21:48:05] [SCAN] Done: 982 files, 905 size groups, 0 skipped by extension
[21:48:05] [HASH] Starting hash comparison for 68 candidate groups
[21:48:05] [HASH] Group 50/68 (size=3761253 bytes, files=2)
[21:48:06] [DONE] Duplicate groups: 54, to remove: 57
Duplicate summary:
- groups: 54
- duplicate_files: 57
- duplicate_bytes: 126243340
- duplicate_gb: 0.118
- planned deletion scope (files): 57

[Duplicate Group 1] Hash=3a392c9b1719... | size=2871542 bytes | files=2
  KEEP: C:\Users\Carina\OneDrive\Laptop\Carina\PhD\Master Certificate.pdf
  DELETE : C:\Users\Carina\OneDrive\Laptop\Carina\Bewerbung\Abschluss  Master.pdf

[Duplicate Group 2] Hash=656fab2e4836... | size=259904 bytes | files=2
  KEEP: C:\Users\Carina\OneDrive\NAS\Carina\Bewerbung\Mappe\Arbeitszeugnis bytes Coding.pdf
  DELETE : C:\Users\Carina\OneDrive\Laptop\Carina\

In [8]:
# 1B) Find and flag similar images (near-duplicates)
log("===== PHASE 1B: FLAG SIMILAR IMAGES =====")

similar_groups = []
similar_plan_actions = []
similar_pairs = []

if FIND_SIMILAR_IMAGES:
    similar_result = find_similar_image_groups(
        ROOT_DIR,
        threshold=SIMILAR_HASH_THRESHOLD,
        neighbor_window=SIMILAR_NEIGHBOR_WINDOW,
        min_file_size_bytes=SIMILAR_MIN_FILE_SIZE_BYTES,
    )

    similar_groups = similar_result.get("groups", [])
    similar_plan_actions = similar_result.get("actions", [])
    similar_pairs = similar_result.get("pairs", [])

    print("Similarity summary:")
    print(f"- gruppen: {len(similar_groups)}")
    print(f"- markierte_dateien: {len(similar_plan_actions)}")
    print(f"- paare: {len(similar_pairs)}")

    if SHOW_PREVIEW_DETAILS:
        limit = None if PREVIEW_MAX_SIMILAR_GROUPS <= 0 else PREVIEW_MAX_SIMILAR_GROUPS
        print_similar_plan(similar_groups, max_groups=limit)
    else:
        print("Detail preview disabled (SHOW_PREVIEW_DETAILS=False).")
else:
    print("FIND_SIMILAR_IMAGES=False -> Similarity search disabled.")


[21:48:06] ===== PHASE 1B: FLAG SIMILAR IMAGES =====
[21:48:06] [SIMILAR] Starting (threshold=6, window=20, min_size=20000 bytes)


C:\Users\Carina\AppData\Local\Temp\ipykernel_12112\3933884275.py:565: DeprecationWarning: Image.Image.getdata is deprecated and will be removed in Pillow 14 (2027-10-15). Use get_flattened_data instead.
  pixels = list(gray.getdata())


[21:48:24] [SIMILAR] Signaturen erstellt: 323 (gescannt=323, zu_klein=0)
[21:48:24] [SIMILAR] Done: groups=48, markierte files=97, pairs=187
Similarity summary:
- gruppen: 48
- markierte_dateien: 97
- paare: 187

[Similarity Group 1] files=10
  REFERENCE: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\03\PXL_20250330_062836973.jpg
  SIMILAR: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\03-1\PXL_20250330_062836973.jpg
  SIMILAR: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\03\PXL_20250330_062837481.jpg
  SIMILAR: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\03-1\PXL_20250330_062837481.jpg
  SIMILAR: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\03\PXL_20250330_062838173.jpg
  SIMILAR: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\03-1\PXL_20250330_062838173.jpg
  SIMILAR: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\03\PXL_20250330_062838319.jpg
  SIMIL

In [9]:
# 1C) AI visual groups (same motif, minimal motion)
# This phase only marks files; it does not delete anything.

globals().setdefault("FIND_AI_VISUAL_GROUPS", True)
globals().setdefault("AI_MODEL_NAME", "openai/clip-vit-base-patch32")
globals().setdefault("AI_SIMILARITY_THRESHOLD", 0.93)
globals().setdefault("AI_SIMILARITY_PERCENT", 92.0)  # Recommended default; set None to use threshold only
globals().setdefault("AI_NEIGHBOR_WINDOW", 60)
globals().setdefault("AI_MIN_FILE_SIZE_BYTES", 20000)
globals().setdefault("AI_BATCH_SIZE", 16)
globals().setdefault("AI_ENABLE_TF32", True)
globals().setdefault("AI_USE_FP16_ON_CUDA", True)
globals().setdefault("AI_DEVICE", "auto")
globals().setdefault("AI_COMPARE_ALL_PAIRS", True)
globals().setdefault("AI_VERBOSE", True)
globals().setdefault("AI_VERBOSE_PAIR_LOG_LIMIT", 25)
globals().setdefault("PREVIEW_MAX_AI_GROUPS", 0)

import numpy as np


def _ai_print_groups(groups: list[dict], max_groups: Optional[int] = None) -> None:
    if not groups:
        print("No AI visual groups found.")
        return

    shown = groups
    if max_groups is not None and max_groups > 0:
        shown = groups[:max_groups]

    for g in shown:
        print(f"\n[AI Group {g['group']}] files={g['count']}")
        print(f"  KEEP (suggested): {g['keep']}")
        for s in g["similar"]:
            print(f"  VERY SIMILAR    : {s}")

    if len(shown) < len(groups):
        print(f"\n... {len(groups) - len(shown)} more AI groups not printed (see CSV report).")


def _load_clip_model(model_name: str):
    try:
        import torch
        from transformers import CLIPModel, CLIPProcessor
    except Exception:
        log("[AI] Missing packages. Install with: pip install torch transformers")
        return None, None, None

    device, _device_id = _resolve_ai_device(torch)
    _configure_ai_torch_backends(torch, device)
    use_fp16 = _use_fp16_for_device(device)
    dtype_label = "fp16" if use_fp16 else "fp32"

    try:
        import contextlib
        import io
        import warnings

        suppress_model_logs = bool(globals().get("AI_SUPPRESS_MODEL_LOAD_WARNINGS", True))
        model_kwargs = {}
        if use_fp16:
            model_kwargs["torch_dtype"] = torch.float16

        if suppress_model_logs:
            with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                warnings.filterwarnings("ignore", message=r".*UNEXPECTED.*")
                model = CLIPModel.from_pretrained(model_name, **model_kwargs)
                processor = CLIPProcessor.from_pretrained(model_name)
        else:
            model = CLIPModel.from_pretrained(model_name, **model_kwargs)
            processor = CLIPProcessor.from_pretrained(model_name)
    except Exception as e:
        log(f"[AI] Failed to load model ({model_name}): {e}")
        return None, None, None

    model = model.to(device)
    if use_fp16 and hasattr(model, "half"):
        model = model.half()
    model.eval()
    log(f"[AI] CLIP loaded: {model_name} on {device} (dtype={dtype_label})")
    return model, processor, torch


def _resolve_similarity_threshold(
        similarity_threshold: float,
        similarity_percent: Optional[float],
) -> float:
    if similarity_percent is not None:
        threshold = float(similarity_percent) / 100.0
    else:
        threshold = float(similarity_threshold)
        if threshold > 1.0:
            threshold = threshold / 100.0

    return max(0.0, min(1.0, threshold))


def find_ai_visual_groups(
        root: Path,
        similarity_threshold: float = 0.93,
        similarity_percent: Optional[float] = None,
        neighbor_window: int = 60,
        min_file_size_bytes: int = 0,
        model_name: str = "openai/clip-vit-base-patch32",
        batch_size: int = 16,
        verbose: bool = False,
        verbose_pair_log_limit: int = 0,
        compare_all_pairs: bool = False,
) -> dict:
    """
    Find burst-like images with very similar motifs via CLIP embeddings.
    Typical use case: repeated shutter shots with minimal movement.
    """
    if Image is None:
        log("[AI] Pillow missing -> AI visual grouping skipped")
        return {"groups": [], "actions": [], "pairs": []}

    model, processor, torch_mod = _load_clip_model(model_name)
    if model is None:
        return {"groups": [], "actions": [], "pairs": []}

    similarity_threshold = _resolve_similarity_threshold(similarity_threshold, similarity_percent)
    neighbor_window = max(1, int(neighbor_window))
    batch_size = max(1, int(batch_size))
    verbose_pair_log_limit = max(0, int(verbose_pair_log_limit))
    compare_all_pairs = bool(compare_all_pairs)

    log(
        f"[AI] Starting visual grouping "
        f"(threshold={similarity_threshold:.4f} / {similarity_threshold * 100:.1f}%, "
        f"window={neighbor_window}, min_size={min_file_size_bytes}, verbose={verbose}, "
        f"compare_all={compare_all_pairs})"
    )

    candidates = []
    scan_stats = {
        "all_files": 0,
        "non_images": 0,
        "image_files": 0,
        "stat_errors": 0,
        "too_small": 0,
        "candidates": 0,
    }

    current_folder = None
    for p in iter_files(root, EXCLUDE_DIR_NAMES, follow_symlinks=FOLLOW_SYMLINKS):
        scan_stats["all_files"] += 1

        folder = str(p.parent)
        if verbose and folder != current_folder:
            current_folder = folder
            log(f"[AI][SCAN] Checking folder: {folder}")

        if p.suffix.lower() not in IMAGE_EXTENSIONS:
            scan_stats["non_images"] += 1
            continue

        scan_stats["image_files"] += 1
        if LOG_EVERY_FILES and scan_stats["image_files"] % LOG_EVERY_FILES == 0:
            log(f"[AI][SCAN] Image files checked: {scan_stats['image_files']:,}")

        try:
            st = p.stat()
        except OSError:
            scan_stats["stat_errors"] += 1
            if verbose:
                log(f"[AI][SCAN] Failed to stat file: {p}")
            continue

        if st.st_size < min_file_size_bytes:
            scan_stats["too_small"] += 1
            continue

        dt = get_image_datetime(p)
        ts = dt.timestamp() if dt else st.st_mtime
        candidates.append({"path": p, "timestamp": ts, "size": st.st_size, "mtime": st.st_mtime})
        scan_stats["candidates"] += 1

    if verbose:
        log(
            "[AI][SCAN] Summary: "
            f"files={scan_stats['all_files']:,}, "
            f"images={scan_stats['image_files']:,}, "
            f"candidates={scan_stats['candidates']:,}, "
            f"too_small={scan_stats['too_small']:,}, "
            f"stat_errors={scan_stats['stat_errors']:,}"
        )

    if len(candidates) < 2:
        log("[AI] Too few image candidates")
        return {"groups": [], "actions": [], "pairs": []}

    device = next(model.parameters()).device
    rows = []

    def _unwrap_clip_features(raw):
        if hasattr(raw, "image_embeds") and raw.image_embeds is not None:
            return raw.image_embeds
        if hasattr(raw, "pooler_output") and raw.pooler_output is not None:
            return raw.pooler_output
        if hasattr(raw, "last_hidden_state") and raw.last_hidden_state is not None:
            hidden = raw.last_hidden_state
            if getattr(hidden, "ndim", 0) >= 3:
                return hidden[:, 0, :]
            return hidden
        if isinstance(raw, (tuple, list)) and raw:
            return raw[0]
        return raw

    total_batches = (len(candidates) + batch_size - 1) // batch_size
    decode_errors_total = 0

    for batch_idx, start in enumerate(range(0, len(candidates), batch_size), start=1):
        chunk = candidates[start:start + batch_size]
        if verbose:
            log(f"[AI][EMBED] Batch {batch_idx}/{total_batches}: loading {len(chunk)} candidates")

        imgs = []
        meta = []
        decode_errors_batch = 0

        for row in chunk:
            try:
                with Image.open(row["path"]) as im:
                    imgs.append(im.convert("RGB"))
                meta.append(row)
            except Exception:
                decode_errors_batch += 1
                continue

        if decode_errors_batch:
            decode_errors_total += decode_errors_batch
            if verbose:
                log(f"[AI][EMBED] Batch {batch_idx}: decode errors={decode_errors_batch}")

        if not meta:
            if verbose:
                log(f"[AI][EMBED] Batch {batch_idx}: no valid images after decode")
            continue

        inputs = processor(images=imgs, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        use_fp16 = str(device).startswith("cuda") and bool(globals().get("AI_USE_FP16_ON_CUDA", True))
        with torch_mod.inference_mode():
            if use_fp16 and hasattr(torch_mod, "autocast"):
                with torch_mod.autocast(device_type="cuda", dtype=torch_mod.float16):
                    feats_raw = model.get_image_features(**inputs)
            else:
                feats_raw = model.get_image_features(**inputs)
            feats = _unwrap_clip_features(feats_raw)

        if feats is None or not hasattr(feats, "norm"):
            log(
                f"[AI][EMBED] Unexpected feature output ({type(feats).__name__}) "
                f"for batch {batch_idx}; skipping batch"
            )
            continue

        feats = feats / feats.norm(dim=-1, keepdim=True).clamp_min(1e-12)
        arr = feats.detach().cpu().numpy().astype(np.float32)
        for m, emb in zip(meta, arr):
            r = dict(m)
            r["embedding"] = emb
            rows.append(r)

        if verbose:
            log(f"[AI][EMBED] Batch {batch_idx}: embeddings added={len(meta)}")

    if verbose:
        log(f"[AI][EMBED] Summary: embeddings={len(rows):,}, decode_errors={decode_errors_total:,}")

    if len(rows) < 2:
        log("[AI] Too few valid embeddings")
        return {"groups": [], "actions": [], "pairs": []}

    rows = sorted(rows, key=lambda r: (r["timestamp"], str(r["path"]).lower()))

    pairs = []
    comparisons = 0
    if verbose:
        log(f"[AI][PAIR] Starting pair search over {len(rows):,} embeddings")

    for i, a in enumerate(rows):
        if LOG_EVERY_FILES and i > 0 and i % LOG_EVERY_FILES == 0:
            log(f"[AI][PAIR] Progress: {i:,}/{len(rows):,}")

        end = len(rows) if compare_all_pairs else min(len(rows), i + 1 + neighbor_window)
        for j in range(i + 1, end):
            b = rows[j]
            comparisons += 1
            sim = float(np.dot(a["embedding"], b["embedding"]))
            if sim >= similarity_threshold:
                pair = {
                    "path_a": str(a["path"]),
                    "path_b": str(b["path"]),
                    "similarity": round(sim, 5),
                }
                pairs.append(pair)

                if verbose and verbose_pair_log_limit > 0 and len(pairs) <= verbose_pair_log_limit:
                    log(
                        f"[AI][PAIR] match#{len(pairs)} sim={pair['similarity']:.5f} "
                        f"| {pair['path_a']} <-> {pair['path_b']}"
                    )

    if verbose:
        log(f"[AI][PAIR] Summary: comparisons={comparisons:,}, pairs={len(pairs):,}")

    if not pairs:
        log("[AI] No AI-similar pairs found")
        return {"groups": [], "actions": [], "pairs": []}

    parent: Dict[str, str] = {}

    def uf_find(x: str) -> str:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def uf_union(a: str, b: str) -> None:
        ra = uf_find(a)
        rb = uf_find(b)
        if ra != rb:
            parent[rb] = ra

    for pair in pairs:
        a = pair["path_a"]
        b = pair["path_b"]
        if a not in parent:
            parent[a] = a
        if b not in parent:
            parent[b] = b
        uf_union(a, b)

    components: Dict[str, List[str]] = defaultdict(list)
    for node in parent:
        components[uf_find(node)].append(node)

    row_by_path = {str(r["path"]): r for r in rows}

    def keep_key(path_str: str):
        # Keep suggestion: larger file first, then older mtime, then shorter path.
        rr = row_by_path.get(path_str)
        if rr is None:
            return (float("inf"), float("inf"), len(path_str), path_str.lower())
        return (-rr.get("size", 0), rr.get("mtime", float("inf")), len(path_str), path_str.lower())

    groups = []
    actions = []

    for gid, paths in enumerate(sorted(components.values(), key=lambda g: len(g), reverse=True), start=1):
        if len(paths) < 2:
            continue

        sorted_paths = sorted(paths, key=keep_key)
        keep = sorted_paths[0]
        similar = sorted_paths[1:]

        groups.append({
            "group": gid,
            "keep": keep,
            "similar": similar,
            "count": len(sorted_paths),
        })

        for s in similar:
            actions.append({
                "group": gid,
                "action": "flag_ai_similar",
                "keep": keep,
                "similar": s,
                "status": "flagged",
            })

    log(f"[AI] Done: groups={len(groups):,}, flagged_files={len(actions):,}, pairs={len(pairs):,}")
    return {"groups": groups, "actions": actions, "pairs": pairs}


log("===== PHASE 1C: AI VISUAL GROUPING =====")
ai_visual_groups = []
ai_visual_plan_actions = []
ai_visual_pairs = []

find_ai_visual_groups_enabled = bool(globals().get("FIND_AI_VISUAL_GROUPS", True))
root_for_ai = globals().get("ROOT_DIR")
if root_for_ai is None:
    root_for_ai = Path.cwd()
    log(f"[AI] ROOT_DIR not defined, using current directory: {root_for_ai}")

ai_similarity_threshold = globals().get("AI_SIMILARITY_THRESHOLD", 0.93)
ai_similarity_percent = globals().get("AI_SIMILARITY_PERCENT", None)
ai_neighbor_window = globals().get("AI_NEIGHBOR_WINDOW", 60)
ai_min_file_size_bytes = globals().get("AI_MIN_FILE_SIZE_BYTES", 0)
ai_model_name = globals().get("AI_MODEL_NAME", "openai/clip-vit-base-patch32")
ai_batch_size = globals().get("AI_BATCH_SIZE", 16)
ai_verbose = bool(globals().get("AI_VERBOSE", False))
ai_verbose_pair_log_limit = globals().get("AI_VERBOSE_PAIR_LOG_LIMIT", 0)
ai_compare_all_pairs = bool(globals().get("AI_COMPARE_ALL_PAIRS", False))
show_preview_details = bool(globals().get("SHOW_PREVIEW_DETAILS", True))
preview_max_ai_groups = globals().get("PREVIEW_MAX_AI_GROUPS", 0)

if find_ai_visual_groups_enabled:
    ai_result = find_ai_visual_groups(
        root_for_ai,
        similarity_threshold=ai_similarity_threshold,
        similarity_percent=ai_similarity_percent,
        neighbor_window=ai_neighbor_window,
        min_file_size_bytes=ai_min_file_size_bytes,
        model_name=ai_model_name,
        batch_size=ai_batch_size,
        verbose=ai_verbose,
        verbose_pair_log_limit=ai_verbose_pair_log_limit,
        compare_all_pairs=ai_compare_all_pairs,
    )

    ai_visual_groups = ai_result.get("groups", [])
    ai_visual_plan_actions = ai_result.get("actions", [])
    ai_visual_pairs = ai_result.get("pairs", [])

    print("AI visual summary:")
    print(f"- groups: {len(ai_visual_groups)}")
    print(f"- flagged_files: {len(ai_visual_plan_actions)}")
    print(f"- pairs: {len(ai_visual_pairs)}")

    if show_preview_details:
        limit = None if preview_max_ai_groups <= 0 else preview_max_ai_groups
        _ai_print_groups(ai_visual_groups, max_groups=limit)
    else:
        print("Detail preview disabled (SHOW_PREVIEW_DETAILS=False).")
else:
    print("FIND_AI_VISUAL_GROUPS=False -> AI visual grouping disabled.")


[21:48:24] ===== PHASE 1C: AI VISUAL GROUPING =====


C:\Users\Carina\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


[21:48:31] [AI] CLIP loaded: openai/clip-vit-base-patch32 on cuda (dtype=fp16)
[21:48:31] [AI] Starting visual grouping (threshold=0.9200 / 92.0%, window=60, min_size=20000, verbose=True, compare_all=True)
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\Bewerbung
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\MBA
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\MBA\new\Additional Documents
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\MBA\new\Humanized Version
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\MBA\new\Original Version
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\PhD
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Carina\Wallpaper
[21:48:31] [AI][SCAN] Checking folder: C:\Users\Carina\OneDrive\Laptop\Work\

In [10]:
# 1D) Move AI groups to quarantine (keep 1, move the rest)
log("===== PHASE 1D: AI QUARANTINE =====")

ai_quarantine_plan_actions = []
ai_quarantine_exec_actions = []

ai_groups = globals().get("ai_visual_groups", [])
if not ai_groups:
    print("No AI groups available. Please run PHASE 1C first.")
else:
    ai_quarantine_plan_actions = build_ai_quarantine_plan(ai_groups, ROOT_DIR, AI_QUARANTINE_DIR)
    print(f"Planned AI quarantine actions: {len(ai_quarantine_plan_actions)}")

    if SHOW_PREVIEW_DETAILS:
        limit = None if PREVIEW_MAX_AI_QUARANTINE <= 0 else PREVIEW_MAX_AI_QUARANTINE
        print_ai_quarantine_plan(ai_quarantine_plan_actions, max_rows=limit)
    else:
        print("Detail preview disabled (SHOW_PREVIEW_DETAILS=False).")

    if MOVE_AI_GROUPS_TO_QUARANTINE:
        print("AI quarantine execution requested.")
        if should_execute_changes():
            ai_quarantine_exec_actions = execute_ai_quarantine(ai_quarantine_plan_actions, dry_run=False)
            moved = sum(1 for r in ai_quarantine_exec_actions if r.get('status') == 'moved')
            errs = sum(1 for r in ai_quarantine_exec_actions if r.get('status') == 'error')
            print(f"Quarantine executed: moved={moved}, error={errs}")
        else:
            print("AI quarantine execution was not started due to safety checks.")
    else:
        print("MOVE_AI_GROUPS_TO_QUARANTINE=False -> Preview only, no move execution.")


[21:49:12] ===== PHASE 1D: AI QUARANTINE =====
Planned AI quarantine actions: 199
[1] GROUP 1 | KEEP: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041433336.jpg
    VERSCHIEBEN: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041704275.jpg
    NACH      : C:\Users\Carina\OneDrive\_duplicate_cleaner_quarantine\ai_visual\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041704275.jpg
[2] GROUP 1 | KEEP: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041433336.jpg
    VERSCHIEBEN: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041722815.jpg
    NACH      : C:\Users\Carina\OneDrive\_duplicate_cleaner_quarantine\ai_visual\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041722815.jpg
[3] GROUP 1 | KEEP: C:\Users\Carina\OneDrive\NAS\MyPhoto\Backup\Carinas Pixel 9\2025\04\PXL_20250427_041433336.jpg
    VERSCHIEBEN: C:\Users\Carina\OneDrive\NAS\M

In [11]:
# 2) Delete duplicates (only after preview + confirmation)
log("===== PHASE 2: DELETE DUPLICATES =====")

delete_actions = []

if DELETE_DUPLICATES:
    print("Delete execution requested.")
    if should_execute_changes():
        delete_actions = delete_duplicates(groups, dry_run=False)
        print(f"Delete actions executed: {len(delete_actions)}")
    else:
        print("Delete execution was not started due to safety checks.")
else:
    print("DELETE_DUPLICATES=False -> No files deleted.")


[21:49:12] ===== PHASE 2: DELETE DUPLICATES =====
DELETE_DUPLICATES=False -> No files deleted.


In [12]:
# 3) Rename image files: preview first, then optional execute
log("===== PHASE 3: RENAME IMAGES =====")

rename_preview_actions = []
rename_actions = []

rename_with_ai_label = bool(globals().get("RENAME_WITH_AI_LABEL", False))
ai_rename_model_name = globals().get("AI_RENAME_MODEL_NAME", "Salesforce/blip-image-captioning-large")
ai_rename_model_fallback = globals().get("AI_RENAME_MODEL_FALLBACK", "Salesforce/blip-image-captioning-base")
ai_rename_max_words = int(globals().get("AI_RENAME_MAX_WORDS", 8))
ai_rename_max_chars = int(globals().get("AI_RENAME_MAX_CHARS", 120))
ai_rename_date_style = str(globals().get("AI_RENAME_DATE_STYLE", "english_text"))
ai_rename_include_time = bool(globals().get("AI_RENAME_INCLUDE_TIME", False))
ai_rename_use_caption_sentence = bool(globals().get("AI_RENAME_USE_CAPTION_SENTENCE", True))
ai_enrich_date_only_names = bool(globals().get("AI_ENRICH_DATE_ONLY_NAMES", True))
rename_all_images_with_ai_label = bool(globals().get("RENAME_ALL_IMAGES_WITH_AI_LABEL", False))
rename_verbose_suggestions = bool(globals().get("RENAME_VERBOSE_SUGGESTIONS", True))
preview_when_disabled = bool(globals().get("PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED", True))

run_preview = bool(RENAME_IMAGES) or preview_when_disabled

if run_preview:
    if not RENAME_IMAGES:
        print("RENAME_IMAGES=False, but preview mode is enabled -> generating rename suggestions only.")

    # Always run preview first.
    rename_preview_actions = rename_cryptic_images(
        ROOT_DIR,
        dry_run=True,
        ai_describe=rename_with_ai_label,
        ai_model_name=ai_rename_model_name,
        ai_model_fallback=ai_rename_model_fallback,
        ai_max_words=ai_rename_max_words,
        ai_max_chars=ai_rename_max_chars,
        rename_date_style=ai_rename_date_style,
        rename_include_time=ai_rename_include_time,
        rename_caption_as_sentence=ai_rename_use_caption_sentence,
        enrich_date_only_names=ai_enrich_date_only_names,
        rename_all_images=rename_all_images_with_ai_label,
        verbose_suggestions=rename_verbose_suggestions,
    )
    print(f"Planned renames: {len(rename_preview_actions)}")

    if SHOW_PREVIEW_DETAILS:
        limit = None if PREVIEW_MAX_RENAMES <= 0 else PREVIEW_MAX_RENAMES
        print_rename_plan(rename_preview_actions, max_rows=limit)
    else:
        print("Detail preview disabled (SHOW_PREVIEW_DETAILS=False).")
else:
    print("Preview disabled: set RENAME_IMAGES=True or PREVIEW_RENAME_SUGGESTIONS_WHEN_RENAME_DISABLED=True.")

if RENAME_IMAGES:
    if should_execute_changes():
        rename_actions = rename_cryptic_images(
            ROOT_DIR,
            dry_run=False,
            ai_describe=rename_with_ai_label,
            ai_model_name=ai_rename_model_name,
            ai_model_fallback=ai_rename_model_fallback,
            ai_max_words=ai_rename_max_words,
            ai_max_chars=ai_rename_max_chars,
            rename_date_style=ai_rename_date_style,
            rename_include_time=ai_rename_include_time,
            rename_caption_as_sentence=ai_rename_use_caption_sentence,
            enrich_date_only_names=ai_enrich_date_only_names,
            rename_all_images=rename_all_images_with_ai_label,
            verbose_suggestions=rename_verbose_suggestions,
        )
        print(f"Executed renames: {len(rename_actions)}")
    else:
        print("Rename execution not started (preview only).")
else:
    print("RENAME_IMAGES=False -> No files renamed (preview can still be generated).")


[21:49:12] ===== PHASE 3: RENAME IMAGES =====
[21:49:12] [RENAME] Start (DRY-RUN)


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-large
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you e

[21:49:17] [RENAME-AI] Caption model loaded: Salesforce/blip-image-captioning-large on cuda (backend=direct_blip, dtype=fp16)
[21:49:18] [PLAN RENAME] found=C:\Users\Carina\OneDrive\Laptop\Carina\Wallpaper\UltrawideWallpapersDotNet-2266.png | current_name=UltrawideWallpapersDotNet-2266.png | proposed_name=06. Feb.2026_A close up of a colorful wave of light on a black background.png | proposed_path=C:\Users\Carina\OneDrive\Laptop\Carina\Wallpaper\06. Feb.2026_A close up of a colorful wave of light on a black background.png | ai_label=close_up_of_colorful_wave_of_light_on_black_background | ai_caption=a close up of a colorful wave of light on a black background
[21:49:18] [PLAN RENAME] found=C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\THM-ANRGT1JXVX.png | current_name=THM-ANRGT1JXVX.png | proposed_name=05. Nov.2022_Certificate of completion of the course.png | proposed_path=C:\Users\Carina\OneDrive\NAS\Carina\Dateien und Texte\Zertifikate\05. Nov.2022_Certificate of

In [13]:
# 5) Write final reports (detailed, decision-oriented)
log("===== PHASE 5: WRITE FINAL REPORTS =====")

report_root = globals().get("ROOT_DIR") or Path.cwd()
report_dir = report_root / "_duplicate_cleaner_reports"
report_dir.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# Fallbacks in case some phases were not executed yet
duplicate_groups = globals().get("groups", [])
duplicate_plan_actions = globals().get("duplicate_plan_actions", [])
similar_groups = globals().get("similar_groups", [])
similar_pairs = globals().get("similar_pairs", [])
ai_visual_groups = globals().get("ai_visual_groups", [])
ai_visual_pairs = globals().get("ai_visual_pairs", [])
rename_preview_actions = globals().get("rename_preview_actions", [])
rename_actions = globals().get("rename_actions", [])


def _name_of(path_like: str) -> str:
    try:
        return Path(str(path_like)).name
    except Exception:
        return str(path_like)


# ------------------------------------------------------------
# Report A: Duplicate decisions (human-readable text)
# ------------------------------------------------------------
duplicate_text_lines = [
    "ZU LOESCHEN WEIL DUPLIKATE DATEI",
    "",
]

duplicate_groups_for_report = []
if duplicate_groups:
    for group_idx, g in enumerate(duplicate_groups, start=1):
        keep_path = str(g.get("keep", ""))
        remove_paths = [str(x) for x in g.get("remove", [])]
        duplicate_groups_for_report.append(
            {
                "group": group_idx,
                "hash": g.get("hash", ""),
                "keep": keep_path,
                "remove": remove_paths,
            }
        )
elif duplicate_plan_actions:
    grouped = defaultdict(list)
    for row in duplicate_plan_actions:
        key = (
            row.get("group", ""),
            str(row.get("kept", "")),
            row.get("hash", ""),
        )
        grouped[key].append(str(row.get("path", "")))

    for idx, (key, paths) in enumerate(grouped.items(), start=1):
        grp, kept, digest = key
        duplicate_groups_for_report.append(
            {
                "group": grp if grp != "" else idx,
                "hash": digest,
                "keep": kept,
                "remove": [str(x) for x in paths],
            }
        )

if duplicate_groups_for_report:
    for g in duplicate_groups_for_report:
        keep_path = g.get("keep", "")
        keep_name = _name_of(keep_path)
        digest = g.get("hash", "")
        remove_paths = g.get("remove", [])

        duplicate_text_lines.append(f"[Duplicate Set {g.get('group', '')}]")
        duplicate_text_lines.append(f"Datei behalten: {keep_name}")
        duplicate_text_lines.append(f"Pfad: {keep_path}")
        duplicate_text_lines.append(f"Hash (SHA-256): {digest}")
        duplicate_text_lines.append("Dateien identisch und zu loeschen:")

        if remove_paths:
            for rp in remove_paths:
                duplicate_text_lines.append(f"- {_name_of(rp)} | {rp}")
        else:
            duplicate_text_lines.append("- (none)")
        duplicate_text_lines.append("")
else:
    duplicate_text_lines.append("No duplicate groups found.")

report_duplicates_text_path = report_dir / f"duplicates_decision_report_{ts}.txt"
report_duplicates_text_path.write_text("\n".join(duplicate_text_lines) + "\n", encoding="utf-8")

# Also write a row-wise duplicate CSV (one row per file to delete)
duplicate_delete_rows = []
for g in duplicate_groups_for_report:
    keep_path = g.get("keep", "")
    keep_name = _name_of(keep_path)
    digest = g.get("hash", "")
    for rp in g.get("remove", []):
        duplicate_delete_rows.append(
            {
                "duplicate_set": g.get("group", ""),
                "keep_file_name": keep_name,
                "keep_file_path": keep_path,
                "keep_file_sha256": digest,
                "delete_file_name": _name_of(rp),
                "delete_file_path": rp,
                "decision": "delete_duplicate",
            }
        )

if not duplicate_delete_rows:
    duplicate_delete_rows = [
        {
            "duplicate_set": "",
            "keep_file_name": "",
            "keep_file_path": "",
            "keep_file_sha256": "",
            "delete_file_name": "",
            "delete_file_path": "",
            "decision": "no_duplicates_found",
        }
    ]

report_duplicates_csv_path = report_dir / f"duplicates_delete_items_{ts}.csv"
save_csv(report_duplicates_csv_path, duplicate_delete_rows)


# ------------------------------------------------------------
# Report B: Similar images (keep/delete recommendation)
# ------------------------------------------------------------
def _pair_key(a: str, b: str) -> tuple[str, str]:
    a = str(a)
    b = str(b)
    return (a, b) if a <= b else (b, a)


dhash_similarity_map = {}
for pair in similar_pairs:
    a = str(pair.get("path_a", ""))
    b = str(pair.get("path_b", ""))
    if not a or not b:
        continue

    dist = pair.get("distance")
    sim_pct = ""
    if dist is not None:
        try:
            sim_pct = round((1.0 - (float(dist) / 64.0)) * 100.0, 2)
        except Exception:
            sim_pct = ""

    k = _pair_key(a, b)
    if k not in dhash_similarity_map:
        dhash_similarity_map[k] = sim_pct

ai_similarity_map = {}
for pair in ai_visual_pairs:
    a = str(pair.get("path_a", ""))
    b = str(pair.get("path_b", ""))
    if not a or not b:
        continue

    sim_pct = ""
    sim = pair.get("similarity")
    if sim is not None:
        try:
            sim_pct = round(float(sim) * 100.0, 2)
        except Exception:
            sim_pct = ""

    k = _pair_key(a, b)
    if k not in ai_similarity_map:
        ai_similarity_map[k] = sim_pct

similarity_rows = []

for g in similar_groups:
    group_id = g.get("group", "")
    keep_path = str(g.get("keep", ""))
    for cand in g.get("similar", []):
        cand_path = str(cand)
        sim_pct = dhash_similarity_map.get(_pair_key(keep_path, cand_path), "")
        similarity_rows.append(
            {
                "source": "dhash",
                "group": group_id,
                "keep_file_name": _name_of(keep_path),
                "keep_file_path": keep_path,
                "candidate_file_name": _name_of(cand_path),
                "candidate_file_path": cand_path,
                "similarity_percent": sim_pct,
                "keep_decision": "keep",
                "candidate_decision": "delete_candidate_after_review",
            }
        )

for g in ai_visual_groups:
    group_id = g.get("group", "")
    keep_path = str(g.get("keep", ""))
    for cand in g.get("similar", []):
        cand_path = str(cand)
        sim_pct = ai_similarity_map.get(_pair_key(keep_path, cand_path), "")
        candidate_decision = "move_to_quarantine" if MOVE_AI_GROUPS_TO_QUARANTINE else "delete_candidate_after_review"
        similarity_rows.append(
            {
                "source": "ai_clip",
                "group": group_id,
                "keep_file_name": _name_of(keep_path),
                "keep_file_path": keep_path,
                "candidate_file_name": _name_of(cand_path),
                "candidate_file_path": cand_path,
                "similarity_percent": sim_pct,
                "keep_decision": "keep",
                "candidate_decision": candidate_decision,
            }
        )

if not similarity_rows:
    similarity_rows = [
        {
            "source": "",
            "group": "",
            "keep_file_name": "",
            "keep_file_path": "",
            "candidate_file_name": "",
            "candidate_file_path": "",
            "similarity_percent": "",
            "keep_decision": "",
            "candidate_decision": "no_similar_images_found",
        }
    ]

report_similarity_path = report_dir / f"similarity_keep_delete_report_{ts}.csv"
save_csv(report_similarity_path, similarity_rows)

# ------------------------------------------------------------
# Report C: Rename plan (original -> new)
# ------------------------------------------------------------
rename_rows = []
rename_source = rename_preview_actions if rename_preview_actions else rename_actions

for idx, a in enumerate(rename_source, start=1):
    src = str(a.get("from") or a.get("path") or "")
    dst = str(a.get("to") or a.get("target") or "")

    rename_rows.append(
        {
            "rename_id": idx,
            "original_file_name": _name_of(src),
            "original_file_path": src,
            "new_file_name": _name_of(dst),
            "new_file_path": dst,
            "directory": str(Path(src).parent) if src else "",
            "status": a.get("status", "planned"),
            "ai_label": a.get("ai_label", ""),
            "ai_caption": a.get("ai_caption", ""),
            "ai_model": a.get("ai_model", ""),
        }
    )

if not rename_rows:
    rename_rows = [
        {
            "rename_id": "",
            "original_file_name": "",
            "original_file_path": "",
            "new_file_name": "",
            "new_file_path": "",
            "directory": "",
            "status": "no_rename_actions_found",
            "ai_label": "",
            "ai_caption": "",
            "ai_model": "",
        }
    ]

report_rename_path = report_dir / f"rename_actions_report_{ts}.csv"
save_csv(report_rename_path, rename_rows)

print(f"Report 1/4: {report_duplicates_text_path} (grouped duplicate decisions)")
print(f"- lines: {len(duplicate_text_lines)}")
print(f"Report 2/4: {report_duplicates_csv_path} (one row per duplicate delete item)")
print(f"- rows: {len(duplicate_delete_rows)}")
print(f"Report 3/4: {report_similarity_path} (similarity keep/delete recommendations)")
print(f"- rows: {len(similarity_rows)}")
print(f"Report 4/4: {report_rename_path} (rename original/new/path)")
print(f"- rows: {len(rename_rows)}")

# Expose report paths for following cells
globals()["final_report_paths"] = [
    str(report_duplicates_text_path),
    str(report_duplicates_csv_path),
    str(report_similarity_path),
    str(report_rename_path),
]


[21:50:38] ===== PHASE 5: WRITE FINAL REPORTS =====
Report 1/4: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\duplicates_decision_report_20260301_215038.txt (grouped duplicate decisions)
- lines: 383
Report 2/4: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\duplicates_delete_items_20260301_215038.csv (one row per duplicate delete item)
- rows: 57
Report 3/4: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\similarity_keep_delete_report_20260301_215038.csv (similarity keep/delete recommendations)
- rows: 296
Report 4/4: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\rename_actions_report_20260301_215038.csv (rename original/new/path)
- rows: 287


In [14]:
# 6B) Final report info
log("===== PHASE 6B: FINAL REPORT INFO =====")

report_paths = globals().get("final_report_paths", [])

if report_paths:
    print("Generated reports:")
    for idx, rp in enumerate(report_paths, start=1):
        print(f"- Report {idx}: {rp}")
else:
    print("No report paths available yet. Run PHASE 5 first.")


[21:50:38] ===== PHASE 6B: FINAL REPORT INFO =====
Generated reports:
- Report 1: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\duplicates_decision_report_20260301_215038.txt
- Report 2: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\duplicates_delete_items_20260301_215038.csv
- Report 3: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\similarity_keep_delete_report_20260301_215038.csv
- Report 4: C:\Users\Carina\OneDrive\_duplicate_cleaner_reports\rename_actions_report_20260301_215038.csv


## Usage
1. Run the **Interactive Notebook Setup** cell and configure values in the form (recommended), or set values manually in the configuration cell.
2. With `DRY_RUN=True`, run PHASE 1, 1B, 1C, 1D, and 3 first (preview only).
3. PHASE 1D shows: keep one file per AI group and move the rest to quarantine.
4. If everything looks correct: `DRY_RUN=False`, `CONFIRM_EXECUTION=True`, `CONFIRM_TEXT="YES_DELETE_AND_RENAME"`.
5. For AI quarantine, also set `MOVE_AI_GROUPS_TO_QUARANTINE=True` and run PHASE 1D.
6. Reports are generated in PHASE 5 and summarized in PHASE 6B.

Note: Quarantine only moves files into `_duplicate_cleaner_quarantine/ai_visual` and does not delete directly.

7. Optional: set `RENAME_ALL_IMAGES_WITH_AI_LABEL=True` to rename all images, not only cryptic/date-only names.
8. `AI_RENAME_MAX_WORDS=0` uses full mini-sentence captions for rename labels.
